# 02. Feature Engineering

В этом ноутбуке мы будем создавать признаки для модели кредитного скоринга на основе результатов EDA.

Основная идея:

- использовать основную таблицу `application_train/application_test`;
- создать новые признаки на уровне клиента;
- добавить агрегированные признаки из дополнительных таблиц;
- собрать итоговые train/test датасеты;
- сохранить готовые признаки в `data/processed`.

Feature engineering делаем после EDA, потому что теперь мы понимаем:

- какие признаки важны;
- где есть пропуски и аномалии;
- какие таблицы связаны через `SK_ID_CURR`;
- какие группы признаков могут быть полезны для модели.

## 1. Imports and settings

Импортируем библиотеки, которые понадобятся для feature engineering.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 2. Paths

Зададим пути к данным проекта.

Мы будем читать сырые данные из `data/raw`, а готовые признаки сохранять в `data/processed`.

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR, PROCESSED_DATA_DIR

(PosixPath('/Users/artem/PycharmProjects/credit-risk-scoring-service/data/raw'),
 PosixPath('/Users/artem/PycharmProjects/credit-risk-scoring-service/data/processed'))

## 3. Load main application data

Загрузим основные таблицы:

- `application_train.csv` — обучающая выборка с `TARGET`;
- `application_test.csv` — тестовая выборка без `TARGET`.

Для feature engineering нам удобно временно объединить train и test в одну таблицу, чтобы одинаково создать признаки для обеих выборок.

In [3]:
application_train = pd.read_csv(RAW_DATA_DIR / "application_train.csv")
application_test = pd.read_csv(RAW_DATA_DIR / "application_test.csv")

application_train.shape, application_test.shape

((307511, 122), (48744, 121))

In [4]:
application_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0000,406597.5000,24700.5000,351000.0000,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.0188,-9461,-637,-3648.0000,-2120,NaN,1,1,0,1,1,0,Laborers,1.0000,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.0830,0.2629,0.1394,0.0247,0.0369,0.9722,0.6192,0.0143,0.0000,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.0220,0.0198,0.0000,0.0000,0.0250,0.0369,0.9722,0.6243,0.0144,0.0000,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.0000,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0000,2.0000,2.0000,2.0000,-1134.0000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,100003,0,Cash loans,F,N,N,0,270000.0000,1293502.5000,35698.5000,1129500.0000,Family,State servant,Higher education,Married,House / apartment,0.0035,-16765,-1188,-1186.0000,-291,NaN,1,1,0,1,1,0,Core staff,2.0000,1,1,MONDAY,11,0,0,0,0,0,0,School,0.3113,0.6222,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.0800,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.0790,0.0554,0.0000,0.0000,0.0968,0.0529,0.9851,0.7987,0.0608,0.0800,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.0100,reg oper account,block of flats,0.0714,Block,No,1.0000,0.0000,1.0000,0.0000,-828.0000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,100004,0,Revolving loans,M,Y,Y,0,67500.0000,135000.0000,6750.0000,135000.0000,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.0100,-19046,-225,-4260.0000,-2531,26.0000,1,1,1,1,1,0,Laborers,1.0000,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.5559,0.

In [5]:
application_test.head()

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0000,568800.0000,20560.5000,450000.0000,Unaccompanied,Working,Higher education,Married,House / apartment,0.0188,-19241,-2329,-5170.0000,-812,NaN,1,1,0,1,0,1,NaN,2.0000,2,2,TUESDAY,18,0,0,0,0,0,0,Kindergarten,0.7526,0.7897,0.1595,0.0660,0.0590,0.9732,NaN,NaN,NaN,0.1379,0.1250,NaN,NaN,NaN,0.0505,NaN,NaN,0.0672,0.0612,0.9732,NaN,NaN,NaN,0.1379,0.1250,NaN,NaN,NaN,0.0526,NaN,NaN,0.0666,0.0590,0.9732,NaN,NaN,NaN,0.1379,0.1250,NaN,NaN,NaN,0.0514,NaN,NaN,NaN,block of flats,0.0392,"Stone, brick",No,0.0000,0.0000,0.0000,0.0000,-1740.0000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,100005,Cash loans,M,N,Y,0,99000.0000,222768.0000,17370.0000,180000.0000,Unaccompanied,Working,Secondary / secondary special,Married,House / apartment,0.0358,-18064,-4469,-9118.0000,-1623,NaN,1,1,0,1,0,0,Low-skill Laborers,2.0000,2,2,FRIDAY,9,0,0,0,0,0,0,Self-employed,0.5650,0.2917,0.4330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0.0000,3.0000
2,100013,Cash loans,M,Y,Y,0,202500.0000,663264.0000,69777.0000,630000.0000,NaN,Working,Higher education,Married,House / apartment,0.0191,-20038,-4458,-2175.0000,-3503,5.0000,1,1,0,1,0,0,Drivers,2.0000,2,2,MONDAY,14,0,0,0,0,0,0,Transport: type 3,NaN,0.6998,0.6110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,-856.0000,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,1

## 4. Combine train and test

Для feature engineering временно объединим train и test в одну таблицу.

Это нужно, чтобы:

- одинаково обработать признаки в train и test;
- не дублировать код;
- избежать ситуации, когда в train и test появились разные наборы колонок.

Перед объединением добавим техническую колонку `DATASET`, чтобы потом снова разделить данные.

In [6]:
application_train["DATASET"] = "train"
application_test["DATASET"] = "test"

application_test["TARGET"] = np.nan

application = pd.concat(
    [application_train, application_test],
    axis=0,
    ignore_index=True
)

application.shape

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/3194636754.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train["DATASET"] = "train"
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/3194636754.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_test["DATASET"] = "test"
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/3194636754.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, wh

(356255, 123)

In [7]:
application["DATASET"].value_counts()

DATASET
train    307511
test      48744
Name: count, dtype: int64

In [8]:
application.groupby("DATASET")["TARGET"].agg(["count", "mean"])

,count,mean
DATASET,,
test,0,NaN
train,307511,0.0807


## 5. Basic feature groups

Перед созданием новых признаков разделим колонки на группы:

- идентификаторы;
- технические колонки;
- числовые признаки;
- категориальные признаки.

Это поможет понимать, с какими типами данных мы работаем.

In [9]:
id_cols = ["SK_ID_CURR"]
target_col = "TARGET"
technical_cols = ["DATASET"]

categorical_cols = application.select_dtypes(include="object").columns.tolist()
numeric_cols = application.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Убираем служебные признаки из списков
for col in id_cols + [target_col]:
    if col in numeric_cols:
        numeric_cols.remove(col)

for col in technical_cols:
    if col in categorical_cols:
        categorical_cols.remove(col)

len(numeric_cols), len(categorical_cols)

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/3732252878.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = application.select_dtypes(include="object").columns.tolist()


(104, 16)

In [10]:
categorical_cols

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

In [11]:
numeric_cols[:30]

['CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3']

## 6. Application features

Начнём с признаков из основной таблицы `application`.

В EDA мы видели, что колонка `DAYS_EMPLOYED` содержит аномальное значение `365243`.

Это техническое значение, которое означает отсутствие информации о стаже. Перед созданием признаков заменим его на `NaN`.

Также создадим отдельный флаг `DAYS_EMPLOYED_ANOM`, чтобы модель могла использовать сам факт такой аномалии.

In [12]:
application["DAYS_EMPLOYED_ANOM"] = (application["DAYS_EMPLOYED"] == 365243).astype(int)

application["DAYS_EMPLOYED"] = application["DAYS_EMPLOYED"].replace(365243, np.nan)

application["DAYS_EMPLOYED_ANOM"].value_counts(normalize=True)

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/2305240819.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["DAYS_EMPLOYED_ANOM"] = (application["DAYS_EMPLOYED"] == 365243).astype(int)


DAYS_EMPLOYED_ANOM
0   0.8185
1   0.1815
Name: proportion, dtype: float64

In [13]:
application["DAYS_EMPLOYED"].describe()

count   291607.0000
mean     -2396.6989
std       2334.4800
min     -17912.0000
25%      -3200.0000
50%      -1663.0000
75%       -780.0000
max          0.0000
Name: DAYS_EMPLOYED, dtype: float64

### 6.1 Age and employment features

В исходных данных признаки с префиксом `DAYS_` заданы в днях относительно даты заявки.

Например:

- `DAYS_BIRTH` — возраст клиента в днях, значение отрицательное;
- `DAYS_EMPLOYED` — стаж работы в днях, значение отрицательное;
- `DAYS_REGISTRATION` — сколько дней назад была изменена регистрация;
- `DAYS_ID_PUBLISH` — сколько дней назад был выдан/изменён документ.

Для удобства создадим более интерпретируемые признаки в годах.

In [14]:
application["AGE_YEARS"] = -application["DAYS_BIRTH"] / 365
application["EMPLOYED_YEARS"] = -application["DAYS_EMPLOYED"] / 365
application["REGISTRATION_YEARS"] = -application["DAYS_REGISTRATION"] / 365
application["ID_PUBLISH_YEARS"] = -application["DAYS_ID_PUBLISH"] / 365

application[
    [
        "DAYS_BIRTH",
        "AGE_YEARS",
        "DAYS_EMPLOYED",
        "EMPLOYED_YEARS",
        "DAYS_REGISTRATION",
        "REGISTRATION_YEARS",
        "DAYS_ID_PUBLISH",
        "ID_PUBLISH_YEARS",
    ]
].head()

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/2395966527.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["AGE_YEARS"] = -application["DAYS_BIRTH"] / 365
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/2395966527.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["EMPLOYED_YEARS"] = -application["DAYS_EMPLOYED"] / 365
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/2395966527.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually

,DAYS_BIRTH,AGE_YEARS,DAYS_EMPLOYED,EMPLOYED_YEARS,DAYS_REGISTRATION,REGISTRATION_YEARS,DAYS_ID_PUBLISH,ID_PUBLISH_YEARS
0,-9461,25.9205,-637.0000,1.7452,-3648.0000,9.9945,-2120,5.8082
1,-16765,45.9315,-1188.0000,3.2548,-1186.0000,3.2493,-291,0.7973
2,-19046,52.1808,-225.0000,0.6164,-4260.0000,11.6712,-2531,6.9342
3,-19005,52.0685,-3039.0000,8.3260,-9833.0000,26.9397,-2437,6.6767
4,-19932,54.6082,-3038.0000,8.3233,-4311.0000,11.8110,-3458,9.4740


In [15]:
application[
    [
        "AGE_YEARS",
        "EMPLOYED_YEARS",
        "REGISTRATION_YEARS",
        "ID_PUBLISH_YEARS",
    ]
].describe()

,AGE_YEARS,EMPLOYED_YEARS,REGISTRATION_YEARS,ID_PUBLISH_YEARS
count,356255.0000,291607.0000,356255.0000,356255.0000
mean,43.9486,6.5663,13.6537,8.2249
std,11.9419,6.3958,9.6629,4.1586
min,20.1041,-0.0000,-0.0000,0.0000
25%,34.0411,2.1370,5.4658,4.7041
50%,43.1644,4.5562,12.3342,8.9096
75%,53.9068,8.7671,20.4849,11.8301
max,69.1205,49.0740,67.5945,19.7178


Созданные признаки стали более понятными:

- `AGE_YEARS` — возраст клиента в годах;
- `EMPLOYED_YEARS` — стаж работы в годах;
- `REGISTRATION_YEARS` — сколько лет назад была регистрация;
- `ID_PUBLISH_YEARS` — сколько лет назад был выдан или обновлён документ.

Такие признаки проще интерпретировать, и модель может использовать их вместо сырых отрицательных `DAYS_*` значений.

### 6.2 Amount ratio features

Создадим признаки на основе денежных колонок.

В кредитном скоринге часто важны не только абсолютные суммы, но и отношения между ними:

- насколько сумма кредита велика относительно дохода;
- насколько аннуитетный платёж велик относительно дохода;
- насколько стоимость товара отличается от суммы кредита;
- насколько клиент финансово нагружен ежемесячным платежом.

Такие признаки помогают модели лучше оценивать долговую нагрузку клиента.

In [16]:
application["CREDIT_TO_INCOME_RATIO"] = (
    application["AMT_CREDIT"] / application["AMT_INCOME_TOTAL"]
)

application["ANNUITY_TO_INCOME_RATIO"] = (
    application["AMT_ANNUITY"] / application["AMT_INCOME_TOTAL"]
)

application["CREDIT_TO_ANNUITY_RATIO"] = (
    application["AMT_CREDIT"] / application["AMT_ANNUITY"]
)

application["GOODS_TO_CREDIT_RATIO"] = (
    application["AMT_GOODS_PRICE"] / application["AMT_CREDIT"]
)

application[
    [
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_GOODS_PRICE",
        "CREDIT_TO_INCOME_RATIO",
        "ANNUITY_TO_INCOME_RATIO",
        "CREDIT_TO_ANNUITY_RATIO",
        "GOODS_TO_CREDIT_RATIO",
    ]
].head()

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/3913165158.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["CREDIT_TO_INCOME_RATIO"] = (
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/3913165158.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["ANNUITY_TO_INCOME_RATIO"] = (
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/3913165158.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 

,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,CREDIT_TO_INCOME_RATIO,ANNUITY_TO_INCOME_RATIO,CREDIT_TO_ANNUITY_RATIO,GOODS_TO_CREDIT_RATIO
0,202500.0000,406597.5000,24700.5000,351000.0000,2.0079,0.1220,16.4611,0.8633
1,270000.0000,1293502.5000,35698.5000,1129500.0000,4.7908,0.1322,36.2341,0.8732
2,67500.0000,135000.0000,6750.0000,135000.0000,2.0000,0.1000,20.0000,1.0000
3,135000.0000,312682.5000,29686.5000,297000.0000,2.3162,0.2199,10.5328,0.9498
4,121500.0000,513000.0000,21865.5000,513000.0000,4.2222,0.1800,23.4616,1.0000


In [17]:
amount_ratio_features = [
    "CREDIT_TO_INCOME_RATIO",
    "ANNUITY_TO_INCOME_RATIO",
    "CREDIT_TO_ANNUITY_RATIO",
    "GOODS_TO_CREDIT_RATIO",
]

application[amount_ratio_features].describe()

,CREDIT_TO_INCOME_RATIO,ANNUITY_TO_INCOME_RATIO,CREDIT_TO_ANNUITY_RATIO,GOODS_TO_CREDIT_RATIO
count,356255.0000,356219.0000,356219.0000,355977.0000
mean,3.8495,0.1812,21.0053,0.8997
std,2.6350,0.0947,7.7834,0.0961
min,0.0048,0.0002,8.0367,0.1667
25%,2.0000,0.1149,14.9107,0.8347
50%,3.1589,0.1632,20.0000,0.8938
75%,5.0000,0.2292,26.2604,1.0000
max,84.7368,2.0247,45.3051,6.6667


In [18]:
np.isinf(application[amount_ratio_features]).sum()

CREDIT_TO_INCOME_RATIO     0
ANNUITY_TO_INCOME_RATIO    0
CREDIT_TO_ANNUITY_RATIO    0
GOODS_TO_CREDIT_RATIO      0
dtype: int64

### 6.3 Family and income features

Создадим признаки, связанные с семьёй клиента и доходом на одного члена семьи.

В кредитном скоринге важно учитывать не только общий доход, но и то, сколько людей зависит от этого дохода.

Например, два клиента с одинаковым доходом могут иметь разную финансовую нагрузку, если у одного клиента большая семья, а у другого нет детей.

In [19]:
application["INCOME_PER_PERSON"] = (
    application["AMT_INCOME_TOTAL"] / application["CNT_FAM_MEMBERS"]
)

application["INCOME_PER_CHILD"] = (
    application["AMT_INCOME_TOTAL"] / (application["CNT_CHILDREN"] + 1)
)

application["CHILDREN_RATIO"] = (
    application["CNT_CHILDREN"] / application["CNT_FAM_MEMBERS"]
)

application["CREDIT_PER_PERSON"] = (
    application["AMT_CREDIT"] / application["CNT_FAM_MEMBERS"]
)

family_income_features = [
    "INCOME_PER_PERSON",
    "INCOME_PER_CHILD",
    "CHILDREN_RATIO",
    "CREDIT_PER_PERSON",
]

application[
    [
        "AMT_INCOME_TOTAL",
        "CNT_CHILDREN",
        "CNT_FAM_MEMBERS",
        *family_income_features,
    ]
].head()

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/822046604.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["INCOME_PER_PERSON"] = (
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/822046604.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["INCOME_PER_CHILD"] = (
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/822046604.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which ha

,AMT_INCOME_TOTAL,CNT_CHILDREN,CNT_FAM_MEMBERS,INCOME_PER_PERSON,INCOME_PER_CHILD,CHILDREN_RATIO,CREDIT_PER_PERSON
0,202500.0000,0,1.0000,202500.0000,202500.0000,0.0000,406597.5000
1,270000.0000,0,2.0000,135000.0000,270000.0000,0.0000,646751.2500
2,67500.0000,0,1.0000,67500.0000,67500.0000,0.0000,135000.0000
3,135000.0000,0,2.0000,67500.0000,135000.0000,0.0000,156341.2500
4,121500.0000,0,1.0000,121500.0000,121500.0000,0.0000,513000.0000


In [20]:
application[family_income_features].describe()

,INCOME_PER_PERSON,INCOME_PER_CHILD,CHILDREN_RATIO,CREDIT_PER_PERSON
count,356253.0000,356255.0000,356253.0000,356253.0000
mean,93786.2963,140754.5088,0.1247,317290.1834
std,98100.8570,140236.9357,0.1990,254473.4279
min,2812.5000,3000.0000,0.0000,6750.0000
25%,49500.0000,78750.0000,0.0000,135000.0000
50%,75000.0000,121500.0000,0.0000,251730.0000
75%,112500.0000,180000.0000,0.3333,422705.2500
max,39000000.0000,58500000.0000,0.9524,4031032.5000


In [21]:
np.isinf(application[family_income_features]).sum()

INCOME_PER_PERSON    0
INCOME_PER_CHILD     0
CHILDREN_RATIO       0
CREDIT_PER_PERSON    0
dtype: int64

### 6.4 EXT_SOURCE aggregation features

В EDA мы увидели, что признаки `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3` имеют сильную связь с `TARGET`.

Создадим агрегированные признаки по этим трём колонкам:

- среднее значение внешних скорингов;
- минимальное значение;
- максимальное значение;
- стандартное отклонение;
- количество пропущенных `EXT_SOURCE`.

Идея: модель сможет использовать не только каждый внешний скор отдельно, но и общий уровень внешней оценки клиента.

In [22]:
ext_source_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

application["EXT_SOURCE_MEAN"] = application[ext_source_cols].mean(axis=1)
application["EXT_SOURCE_MIN"] = application[ext_source_cols].min(axis=1)
application["EXT_SOURCE_MAX"] = application[ext_source_cols].max(axis=1)
application["EXT_SOURCE_STD"] = application[ext_source_cols].std(axis=1)
application["EXT_SOURCE_MISSING_COUNT"] = application[ext_source_cols].isna().sum(axis=1)

ext_source_features = [
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MIN",
    "EXT_SOURCE_MAX",
    "EXT_SOURCE_STD",
    "EXT_SOURCE_MISSING_COUNT",
]

application[
    ext_source_cols + ext_source_features
].head()

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/1919437578.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["EXT_SOURCE_MEAN"] = application[ext_source_cols].mean(axis=1)
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/1919437578.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["EXT_SOURCE_MIN"] = application[ext_source_cols].min(axis=1)
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/1919437578.py:5: PerformanceWarning: DataFrame is highly fragment

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_MAX,EXT_SOURCE_STD,EXT_SOURCE_MISSING_COUNT
0,0.0830,0.2629,0.1394,0.1618,0.0830,0.2629,0.0920,0
1,0.3113,0.6222,NaN,0.4668,0.3113,0.6222,0.2199,1
2,NaN,0.5559,0.7296,0.6427,0.5559,0.7296,0.1228,1
3,NaN,0.6504,NaN,0.6504,0.6504,0.6504,NaN,2
4,NaN,0.3227,NaN,0.3227,0.3227,0.3227,NaN,2


In [23]:
application[ext_source_features].describe()

,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_MAX,EXT_SOURCE_STD,EXT_SOURCE_MISSING_COUNT
count,356076.0000,356076.0000,356076.0000,315305.0000,356255.0000
mean,0.5089,0.3982,0.6166,0.1510,0.7416
std,0.1485,0.1861,0.1548,0.0990,0.6508
min,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.4141,0.2536,0.5410,0.0731,0.0000
50%,0.5238,0.4014,0.6483,0.1363,1.0000
75%,0.6212,0.5496,0.7252,0.2137,1.0000
max,0.8789,0.8789,0.9627,0.6519,3.0000


In [24]:
application.loc[application["DATASET"] == "train"].groupby("TARGET")[
    ext_source_features
].mean()

,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_MAX,EXT_SOURCE_STD,EXT_SOURCE_MISSING_COUNT
TARGET,,,,,
0.0000,0.5191,0.4099,0.6250,0.1499,0.7588
1.0000,0.3970,0.2824,0.5122,0.1676,0.8258


Вывод:

Клиенты с `TARGET = 1` имеют более низкие значения агрегированных `EXT_SOURCE` признаков.

Это подтверждает вывод из EDA: внешние скоринговые признаки являются одними из самых сильных признаков для модели.

Поэтому мы оставляем как исходные признаки `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`, так и новые агрегаты:

- `EXT_SOURCE_MEAN`;
- `EXT_SOURCE_MIN`;
- `EXT_SOURCE_MAX`;
- `EXT_SOURCE_STD`;
- `EXT_SOURCE_MISSING_COUNT`.

### 6.5 Document features

В таблице `application` есть набор бинарных признаков `FLAG_DOCUMENT_*`.

Они показывают наличие разных документов у клиента.

Создадим агрегированные признаки:

- общее количество предоставленных документов;
- среднее значение по документным флагам.

Так модель сможет использовать не только отдельные документные признаки, но и общий уровень заполненности документов.

In [25]:
document_cols = [
    col for col in application.columns
    if col.startswith("FLAG_DOCUMENT_")
]

len(document_cols), document_cols[:5]

(20,
 ['FLAG_DOCUMENT_2',
  'FLAG_DOCUMENT_3',
  'FLAG_DOCUMENT_4',
  'FLAG_DOCUMENT_5',
  'FLAG_DOCUMENT_6'])

In [26]:
application["DOCUMENT_COUNT"] = application[document_cols].sum(axis=1)
application["DOCUMENT_MEAN"] = application[document_cols].mean(axis=1)

document_features = [
    "DOCUMENT_COUNT",
    "DOCUMENT_MEAN",
]

application[
    document_cols[:5] + document_features
].head()

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/1273289131.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["DOCUMENT_COUNT"] = application[document_cols].sum(axis=1)
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/1273289131.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["DOCUMENT_MEAN"] = application[document_cols].mean(axis=1)


,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,DOCUMENT_COUNT,DOCUMENT_MEAN
0,0,1,0,0,0,1,0.0500
1,0,1,0,0,0,1,0.0500
2,0,0,0,0,0,0,0.0000
3,0,1,0,0,0,1,0.0500
4,0,0,0,0,0,1,0.0500


In [27]:
application[document_features].describe()

,DOCUMENT_COUNT,DOCUMENT_MEAN
count,356255.0000,356255.0000
mean,0.9376,0.0469
std,0.3236,0.0162
min,0.0000,0.0000
25%,1.0000,0.0500
50%,1.0000,0.0500
75%,1.0000,0.0500
max,4.0000,0.2000


In [28]:
application.loc[application["DATASET"] == "train"].groupby("TARGET")[
    document_features
].mean()

,DOCUMENT_COUNT,DOCUMENT_MEAN
TARGET,,
0.0000,0.9284,0.0464
1.0000,0.9501,0.0475


Вывод:

Агрегированные документные признаки имеют небольшое различие между `TARGET = 0` и `TARGET = 1`.

Сами по себе они, вероятно, не будут самыми сильными признаками, но могут дать модели дополнительную информацию о клиенте.

### 6.6 Social circle features

В таблице есть признаки, связанные с социальным окружением клиента:

- `OBS_30_CNT_SOCIAL_CIRCLE`;
- `DEF_30_CNT_SOCIAL_CIRCLE`;
- `OBS_60_CNT_SOCIAL_CIRCLE`;
- `DEF_60_CNT_SOCIAL_CIRCLE`.

Они показывают количество наблюдений и дефолтов в окружении клиента.

Создадим ratio-признаки:

- доля дефолтов среди наблюдений за 30 дней;
- доля дефолтов среди наблюдений за 60 дней.

Такие признаки могут отражать рискованность окружения клиента.

In [29]:
application["DEF_30_SOCIAL_RATIO"] = (
    application["DEF_30_CNT_SOCIAL_CIRCLE"] / application["OBS_30_CNT_SOCIAL_CIRCLE"]
)

application["DEF_60_SOCIAL_RATIO"] = (
    application["DEF_60_CNT_SOCIAL_CIRCLE"] / application["OBS_60_CNT_SOCIAL_CIRCLE"]
)

social_features = [
    "DEF_30_SOCIAL_RATIO",
    "DEF_60_SOCIAL_RATIO",
]

application[
    [
        "OBS_30_CNT_SOCIAL_CIRCLE",
        "DEF_30_CNT_SOCIAL_CIRCLE",
        "OBS_60_CNT_SOCIAL_CIRCLE",
        "DEF_60_CNT_SOCIAL_CIRCLE",
        *social_features,
    ]
].head()

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/1352015395.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["DEF_30_SOCIAL_RATIO"] = (
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/1352015395.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["DEF_60_SOCIAL_RATIO"] = (


,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DEF_30_SOCIAL_RATIO,DEF_60_SOCIAL_RATIO
0,2.0000,2.0000,2.0000,2.0000,1.0000,1.0000
1,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000
2,0.0000,0.0000,0.0000,0.0000,NaN,NaN
3,2.0000,0.0000,2.0000,0.0000,0.0000,0.0000
4,0.0000,0.0000,0.0000,0.0000,NaN,NaN


In [30]:
application[social_features].describe()

,DEF_30_SOCIAL_RATIO,DEF_60_SOCIAL_RATIO
count,165270.0000,164399.0000
mean,0.1436,0.1090
std,0.2992,0.2704
min,0.0000,0.0000
25%,0.0000,0.0000
50%,0.0000,0.0000
75%,0.0000,0.0000
max,1.0000,1.0000


In [31]:
np.isinf(application[social_features]).sum()

DEF_30_SOCIAL_RATIO    0
DEF_60_SOCIAL_RATIO    0
dtype: int64

Вывод:

Социальные ratio-признаки созданы корректно.

У большинства клиентов доля дефолтов в социальном окружении равна `0`, но у части клиентов есть ненулевые значения.

Пропуски в новых признаках появляются там, где количество наблюдений в окружении равно `0`. Пока оставляем эти пропуски, обработаем их позже на этапе preprocessing/modeling.

### 6.7 Contact and region features

Создадим несколько агрегированных признаков по контактной информации и региональным флагам.

В исходных данных есть признаки:

- был ли указан мобильный телефон;
- был ли указан рабочий телефон;
- был ли указан email;
- совпадает ли регион проживания и работы;
- совпадает ли город проживания и работы.

Вместо того чтобы смотреть только на отдельные флаги, создадим агрегаты:

- сколько контактных способов указано;
- сколько региональных несовпадений есть у клиента.

In [32]:
contact_cols = [
    "FLAG_MOBIL",
    "FLAG_EMP_PHONE",
    "FLAG_WORK_PHONE",
    "FLAG_CONT_MOBILE",
    "FLAG_PHONE",
    "FLAG_EMAIL",
]

region_mismatch_cols = [
    "REG_REGION_NOT_LIVE_REGION",
    "REG_REGION_NOT_WORK_REGION",
    "LIVE_REGION_NOT_WORK_REGION",
    "REG_CITY_NOT_LIVE_CITY",
    "REG_CITY_NOT_WORK_CITY",
    "LIVE_CITY_NOT_WORK_CITY",
]

application["CONTACT_FLAGS_SUM"] = application[contact_cols].sum(axis=1)
application["REGION_MISMATCH_SUM"] = application[region_mismatch_cols].sum(axis=1)

contact_region_features = [
    "CONTACT_FLAGS_SUM",
    "REGION_MISMATCH_SUM",
]

application[
    contact_cols + region_mismatch_cols + contact_region_features
].head()

/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/1831915388.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["CONTACT_FLAGS_SUM"] = application[contact_cols].sum(axis=1)
/var/folders/b0/dg54wzrd5_97wyrzfmdtzjkm0000gn/T/ipykernel_74552/1831915388.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application["REGION_MISMATCH_SUM"] = application[region_mismatch_cols].sum(axis=1)


,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,CONTACT_FLAGS_SUM,REGION_MISMATCH_SUM
0,1,1,0,1,1,0,0,0,0,0,0,0,4,0
1,1,1,0,1,1,0,0,0,0,0,0,0,4,0
2,1,1,1,1,1,0,0,0,0,0,0,0,5,0
3,1,1,0,1,0,0,0,0,0,0,0,0,3,0
4,1,1,0,1,0,0,0,0,0,0,1,1,3,2


In [33]:
application[contact_region_features].describe()

,CONTACT_FLAGS_SUM,REGION_MISMATCH_SUM
count,356255.0000,356255.0000
mean,3.3666,0.5944
std,0.8697,1.0872
min,1.0000,0.0000
25%,3.0000,0.0000
50%,3.0000,0.0000
75%,4.0000,2.0000
max,6.0000,6.0000


In [34]:
application.loc[application["DATASET"] == "train"].groupby("TARGET")[
    contact_region_features
].mean()

,CONTACT_FLAGS_SUM,REGION_MISMATCH_SUM
TARGET,,
0.0000,3.3498,0.5803
1.0000,3.4158,0.7589


Вывод:

У клиентов с `TARGET = 1` значение `REGION_MISMATCH_SUM` выше, чем у клиентов без дефолта.

Это может означать, что несовпадение региона проживания, работы и регистрации связано с более высоким кредитным риском.

Признаки `CONTACT_FLAGS_SUM` и `REGION_MISMATCH_SUM` оставляем для модели.

## 7. Check created application features

Проверим список новых признаков, которые мы создали на основе основной таблицы `application`.

In [35]:
application_new_features = [
    "DAYS_EMPLOYED_ANOM",
    "AGE_YEARS",
    "EMPLOYED_YEARS",
    "REGISTRATION_YEARS",
    "ID_PUBLISH_YEARS",
    "CREDIT_TO_INCOME_RATIO",
    "ANNUITY_TO_INCOME_RATIO",
    "CREDIT_TO_ANNUITY_RATIO",
    "GOODS_TO_CREDIT_RATIO",
    "INCOME_PER_PERSON",
    "INCOME_PER_CHILD",
    "CHILDREN_RATIO",
    "CREDIT_PER_PERSON",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MIN",
    "EXT_SOURCE_MAX",
    "EXT_SOURCE_STD",
    "EXT_SOURCE_MISSING_COUNT",
    "DOCUMENT_COUNT",
    "DOCUMENT_MEAN",
    "DEF_30_SOCIAL_RATIO",
    "DEF_60_SOCIAL_RATIO",
    "CONTACT_FLAGS_SUM",
    "REGION_MISMATCH_SUM",
]

len(application_new_features)

24

In [36]:
application[application_new_features].head()

,DAYS_EMPLOYED_ANOM,AGE_YEARS,EMPLOYED_YEARS,REGISTRATION_YEARS,ID_PUBLISH_YEARS,CREDIT_TO_INCOME_RATIO,ANNUITY_TO_INCOME_RATIO,CREDIT_TO_ANNUITY_RATIO,GOODS_TO_CREDIT_RATIO,INCOME_PER_PERSON,INCOME_PER_CHILD,CHILDREN_RATIO,CREDIT_PER_PERSON,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_MAX,EXT_SOURCE_STD,EXT_SOURCE_MISSING_COUNT,DOCUMENT_COUNT,DOCUMENT_MEAN,DEF_30_SOCIAL_RATIO,DEF_60_SOCIAL_RATIO,CONTACT_FLAGS_SUM,REGION_MISMATCH_SUM
0,0,25.9205,1.7452,9.9945,5.8082,2.0079,0.1220,16.4611,0.8633,202500.0000,202500.0000,0.0000,406597.5000,0.1618,0.0830,0.2629,0.0920,0,1,0.0500,1.0000,1.0000,4,0
1,0,45.9315,3.2548,3.2493,0.7973,4.7908,0.1322,36.2341,0.8732,135000.0000,270000.0000,0.0000,646751.2500,0.4668,0.3113,0.6222,0.2199,1,1,0.0500,0.0000,0.0000,4,0
2,0,52.1808,0.6164,11.6712,6.9342,2.0000,0.1000,20.0000,1.0000,67500.0000,67500.0000,0.0000,135000.0000,0.6427,0.5559,0.7296,0.1228,1,0,0.0000,NaN,NaN,5,0
3,0,52.0685,8.3260,26.9397,6.6767,2.3162,0.2199,10.5328,0.9498,67500.0000,135000.0000,0.0000,156341.2500,0.6504,0.6504,0.6504,NaN,2,1,0.0500,0.0000,0.0000,3,0
4,0,54.6082,8.3233,11.8110,9.4740,4.2222,0.1800,23.4616,1.0000,121500.0000,121500.0000,0.0000,513000.0000,0.3227,0.3227,0.3227,NaN,2,1,0.0500,NaN,NaN,3,2


In [37]:
application[application_new_features].isna().mean().sort_values(ascending=False)

DEF_60_SOCIAL_RATIO        0.5385
DEF_30_SOCIAL_RATIO        0.5361
EMPLOYED_YEARS             0.1815
EXT_SOURCE_STD             0.1149
GOODS_TO_CREDIT_RATIO      0.0008
EXT_SOURCE_MEAN            0.0005
EXT_SOURCE_MIN             0.0005
EXT_SOURCE_MAX             0.0005
ANNUITY_TO_INCOME_RATIO    0.0001
CREDIT_TO_ANNUITY_RATIO    0.0001
INCOME_PER_PERSON          0.0000
CHILDREN_RATIO             0.0000
CREDIT_PER_PERSON          0.0000
CONTACT_FLAGS_SUM          0.0000
EXT_SOURCE_MISSING_COUNT   0.0000
DOCUMENT_MEAN              0.0000
DOCUMENT_COUNT             0.0000
DAYS_EMPLOYED_ANOM         0.0000
AGE_YEARS                  0.0000
INCOME_PER_CHILD           0.0000
CREDIT_TO_INCOME_RATIO     0.0000
ID_PUBLISH_YEARS           0.0000
REGISTRATION_YEARS         0.0000
REGION_MISMATCH_SUM        0.0000
dtype: float64

Вывод:

На основе основной таблицы `application` создано 24 новых признака.

Часть признаков содержит пропуски. Это ожидаемо, потому что исходные признаки тоже содержали пропущенные значения, а некоторые ratio-признаки создавались через деление.

На этапе feature engineering мы пока не заполняем пропуски и не обрезаем выбросы. Это будет сделано позже на этапе preprocessing/modeling.

## 8. Bureau features

Таблица `bureau.csv` содержит информацию о прошлых кредитах клиента в других финансовых организациях.

Связь с основной таблицей идёт по ключу `SK_ID_CURR`.

Для каждого клиента может быть несколько записей в `bureau`, поэтому перед объединением с `application` нужно агрегировать данные до уровня одного клиента.

In [38]:
application.shape

(356255, 147)

In [39]:
bureau = pd.read_csv(RAW_DATA_DIR / "bureau.csv")

bureau.shape

(1716428, 17)

In [40]:
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0000,-153.0000,NaN,0,91323.0000,0.0000,NaN,0.0000,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0000,NaN,NaN,0,225000.0000,171342.0000,NaN,0.0000,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0000,NaN,NaN,0,464323.5000,NaN,NaN,0.0000,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0000,NaN,NaN,0.0000,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0000,NaN,77674.5000,0,2700000.0000,NaN,NaN,0.0000,Consumer credit,-21,NaN


In [41]:
bureau_records_per_client = (
    bureau.groupby("SK_ID_CURR")["SK_ID_BUREAU"]
    .count()
    .sort_values(ascending=False)
)

bureau_records_per_client.describe()

count   305811.0000
mean         5.6127
std          4.4304
min          1.0000
25%          2.0000
50%          4.0000
75%          8.0000
max        116.0000
Name: SK_ID_BUREAU, dtype: float64

### 8.1 Basic bureau aggregations

Создадим базовые агрегаты по прошлым кредитам клиента.

Для каждого `SK_ID_CURR` посчитаем:

- количество записей в `bureau`;
- количество уникальных прошлых кредитов;
- средние/максимальные/минимальные значения по числовым признакам;
- суммы по просрочкам и задолженностям.

После агрегации одна строка будет соответствовать одному клиенту.

In [42]:
bureau_num_cols = bureau.select_dtypes(include=["int64", "float64"]).columns.tolist()

bureau_num_cols = [
    col for col in bureau_num_cols
    if col not in ["SK_ID_CURR", "SK_ID_BUREAU"]
]

bureau_num_cols

['DAYS_CREDIT',
 'CREDIT_DAY_OVERDUE',
 'DAYS_CREDIT_ENDDATE',
 'DAYS_ENDDATE_FACT',
 'AMT_CREDIT_MAX_OVERDUE',
 'CNT_CREDIT_PROLONG',
 'AMT_CREDIT_SUM',
 'AMT_CREDIT_SUM_DEBT',
 'AMT_CREDIT_SUM_LIMIT',
 'AMT_CREDIT_SUM_OVERDUE',
 'DAYS_CREDIT_UPDATE',
 'AMT_ANNUITY']

In [43]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_RECORDS_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
    BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
    BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),
    BUREAU_CREDIT_DAY_OVERDUE_MEAN=("CREDIT_DAY_OVERDUE", "mean"),
    BUREAU_CREDIT_DAY_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
    BUREAU_DAYS_CREDIT_ENDDATE_MEAN=("DAYS_CREDIT_ENDDATE", "mean"),
    BUREAU_DAYS_CREDIT_UPDATE_MEAN=("DAYS_CREDIT_UPDATE", "mean"),
    BUREAU_AMT_CREDIT_SUM_MEAN=("AMT_CREDIT_SUM", "mean"),
    BUREAU_AMT_CREDIT_SUM_SUM=("AMT_CREDIT_SUM", "sum"),
    BUREAU_AMT_CREDIT_SUM_MAX=("AMT_CREDIT_SUM", "max"),
    BUREAU_AMT_CREDIT_SUM_DEBT_MEAN=("AMT_CREDIT_SUM_DEBT", "mean"),
    BUREAU_AMT_CREDIT_SUM_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN=("AMT_CREDIT_SUM_OVERDUE", "mean"),
    BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_AMT_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    BUREAU_AMT_ANNUITY_SUM=("AMT_ANNUITY", "sum"),
).reset_index()

bureau_agg.shape

(305811, 18)

In [44]:
bureau_agg.head()

,SK_ID_CURR,BUREAU_RECORDS_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_AMT_ANNUITY_SUM
0,100001,7,-735.0000,-1572,-49,0.0000,0,82.4286,-93.1429,207623.5714,1453365.0000,378000.0000,85240.9286,596686.5000,0.0000,0.0000,3545.3571,24817.5000
1,100002,8,-874.0000,-1437,-103,0.0000,0,-349.0000,-499.8750,108131.9456,865055.5650,450000.0000,49156.2000,245781.0000,0.0000,0.0000,0.0000,0.0000
2,100003,4,-1400.7500,-2586,-606,0.0000,0,-544.5000,-816.0000,254350.1250,1017400.5000,810000.0000,0.0000,0.0000,0.0000,0.0000,NaN,0.0000
3,100004,2,-867.0000,-1326,-408,0.0000,0,-488.5000,-532.0000,94518.9000,189037.8000,94537.8000,0.0000,0.0000,0.0000,0.0000,NaN,0.0000
4,100005,3,-190.6667,-373,-62,0.0000,0,439.3333,-54.3333,219042.0000,657126.0000,568800.0000,189469.5000,568408.5000,0.0000,0.0000,1420.5000,4261.5000


In [45]:
bureau_agg.isna().mean().sort_values(ascending=False).head(10)

BUREAU_AMT_ANNUITY_MEAN              0.6134
BUREAU_AMT_CREDIT_SUM_DEBT_MEAN      0.0274
BUREAU_DAYS_CREDIT_ENDDATE_MEAN      0.0085
BUREAU_AMT_CREDIT_SUM_MEAN           0.0000
BUREAU_AMT_CREDIT_SUM_MAX            0.0000
BUREAU_AMT_CREDIT_SUM_SUM            0.0000
BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM    0.0000
BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN   0.0000
BUREAU_AMT_CREDIT_SUM_DEBT_SUM       0.0000
SK_ID_CURR                           0.0000
dtype: float64

### 8.2 Bureau categorical aggregations

Теперь создадим признаки на основе категориальных колонок `bureau`.

Самые важные категориальные признаки:

- `CREDIT_ACTIVE` — статус кредита;
- `CREDIT_TYPE` — тип кредита.

Для них создадим one-hot encoding, а затем агрегируем значения по клиенту.

Так мы получим признаки вроде:

- количество активных кредитов;
- количество закрытых кредитов;
- количество потребительских кредитов;
- количество кредитных карт.

In [46]:
bureau_cat = pd.get_dummies(
    bureau[["SK_ID_CURR", "CREDIT_ACTIVE", "CREDIT_TYPE"]],
    columns=["CREDIT_ACTIVE", "CREDIT_TYPE"],
    dummy_na=True
)

bureau_cat.head()

,SK_ID_CURR,CREDIT_ACTIVE_Active,CREDIT_ACTIVE_Bad debt,CREDIT_ACTIVE_Closed,CREDIT_ACTIVE_Sold,CREDIT_ACTIVE_nan,CREDIT_TYPE_Another type of loan,CREDIT_TYPE_Car loan,CREDIT_TYPE_Cash loan (non-earmarked),CREDIT_TYPE_Consumer credit,CREDIT_TYPE_Credit card,CREDIT_TYPE_Interbank credit,CREDIT_TYPE_Loan for business development,CREDIT_TYPE_Loan for purchase of shares (margin lending),CREDIT_TYPE_Loan for the purchase of equipment,CREDIT_TYPE_Loan for working capital replenishment,CREDIT_TYPE_Microloan,CREDIT_TYPE_Mobile operator loan,CREDIT_TYPE_Mortgage,CREDIT_TYPE_Real estate loan,CREDIT_TYPE_Unknown type of loan,CREDIT_TYPE_nan
0,215354,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
1,215354,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
2,215354,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
3,215354,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
4,215354,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False


In [47]:
bureau_cat_agg = bureau_cat.groupby("SK_ID_CURR").sum().reset_index()

bureau_cat_agg = bureau_cat_agg.rename(
    columns={
        col: f"BUREAU_{col}"
        for col in bureau_cat_agg.columns
        if col != "SK_ID_CURR"
    }
)

bureau_cat_agg.shape

(305811, 22)

In [48]:
bureau_cat_agg.head()

,SK_ID_CURR,BUREAU_CREDIT_ACTIVE_Active,BUREAU_CREDIT_ACTIVE_Bad debt,BUREAU_CREDIT_ACTIVE_Closed,BUREAU_CREDIT_ACTIVE_Sold,BUREAU_CREDIT_ACTIVE_nan,BUREAU_CREDIT_TYPE_Another type of loan,BUREAU_CREDIT_TYPE_Car loan,BUREAU_CREDIT_TYPE_Cash loan (non-earmarked),BUREAU_CREDIT_TYPE_Consumer credit,BUREAU_CREDIT_TYPE_Credit card,BUREAU_CREDIT_TYPE_Interbank credit,BUREAU_CREDIT_TYPE_Loan for business development,BUREAU_CREDIT_TYPE_Loan for purchase of shares (margin lending),BUREAU_CREDIT_TYPE_Loan for the purchase of equipment,BUREAU_CREDIT_TYPE_Loan for working capital replenishment,BUREAU_CREDIT_TYPE_Microloan,BUREAU_CREDIT_TYPE_Mobile operator loan,BUREAU_CREDIT_TYPE_Mortgage,BUREAU_CREDIT_TYPE_Real estate loan,BUREAU_CREDIT_TYPE_Unknown type of loan,BUREAU_CREDIT_TYPE_nan
0,100001,3,0,4,0,0,0,0,0,7,0,0,0,0,0,0,0,0,0,0,0,0
1,100002,2,0,6,0,0,0,0,0,4,4,0,0,0,0,0,0,0,0,0,0,0
2,100003,1,0,3,0,0,0,0,0,2,2,0,0,0,0,0,0,0,0,0,0,0
3,100004,0,0,2,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0
4,100005,2,0,1,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0


In [49]:
bureau_features = bureau_agg.merge(
    bureau_cat_agg,
    on="SK_ID_CURR",
    how="left"
)

bureau_features.shape

(305811, 39)

In [50]:
bureau_features.head()

,SK_ID_CURR,BUREAU_RECORDS_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_AMT_ANNUITY_SUM,BUREAU_CREDIT_ACTIVE_Active,BUREAU_CREDIT_ACTIVE_Bad debt,BUREAU_CREDIT_ACTIVE_Closed,BUREAU_CREDIT_ACTIVE_Sold,BUREAU_CREDIT_ACTIVE_nan,BUREAU_CREDIT_TYPE_Another type of loan,BUREAU_CREDIT_TYPE_Car loan,BUREAU_CREDIT_TYPE_Cash loan (non-earmarked),BUREAU_CREDIT_TYPE_Consumer credit,BUREAU_CREDIT_TYPE_Credit card,BUREAU_CREDIT_TYPE_Interbank credit,BUREAU_CREDIT_TYPE_Loan for business development,BUREAU_CREDIT_TYPE_Loan for purchase of shares (margin lending),BUREAU_CREDIT_TYPE_Loan for the purchase of equipment,BUREAU_CREDIT_TYPE_Loan for working capital replenishment,BUREAU_CREDIT_TYPE_Microloan,BUREAU_CREDIT_TYPE_Mobile operator loan,BUREAU_CREDIT_TYPE_Mortgage,BUREAU_CREDIT_TYPE_Real estate loan,BUREAU_CREDIT_TYPE_Unknown type of loan,BUREAU_CREDIT_TYPE_nan
0,100001,7,-735.0000,-1572,-49,0.0000,0,82.4286,-93.1429,207623.5714,1453365.0000,378000.0000,85240.9286,596686.5000,0.0000,0.0000,3545.3571,24817.5000,3,0,4,0,0,0,0,0,7,0,0,0,0,0,0,0,0,0,0,0,0
1,100002,8,-874.0000,-1437,-103,0.0000,0,-349.0000,-499.8750,108131.9456,865055.5650,450000.0000,49156.2000,245781.0000,0.0000,0.0000,0.0000,0.0000,2,0,6,0,0,0,0,0,4,4,0,0,0,0,0,0,0,0,0,0,0
2,100003,4,-1400.7500,-2586,-606,0.0000,0,-544.5000,-816.0000,254350.1250,1017400.5000,810000.0000,0.0000,0.0000,0.0000,0.0000,NaN,0.0000,1,0,3,0,0,0,0,0,2,2,0,0,0,0,0,0,0,0,0,0,0
3,100004,2,-867.0000,-1326,-408,0.0000,0,-488.5000,-532.0000,94518.9000,189037.8000,94537.8000,0.0000,0.0000,0.0000,0.0000,NaN,0.0000,0,0,2,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0
4,100005,3,-190.6667,-373,-62,0.0000,0,439.3333,-54.3333,219042.0000,657126.0000,568800.0000,189469.5000,568408.5000,0.0000,0.0000,1420.5000,4261.5000,2,0,1,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0


In [51]:
bureau_features.columns.tolist()

['SK_ID_CURR',
 'BUREAU_RECORDS_COUNT',
 'BUREAU_DAYS_CREDIT_MEAN',
 'BUREAU_DAYS_CREDIT_MIN',
 'BUREAU_DAYS_CREDIT_MAX',
 'BUREAU_CREDIT_DAY_OVERDUE_MEAN',
 'BUREAU_CREDIT_DAY_OVERDUE_MAX',
 'BUREAU_DAYS_CREDIT_ENDDATE_MEAN',
 'BUREAU_DAYS_CREDIT_UPDATE_MEAN',
 'BUREAU_AMT_CREDIT_SUM_MEAN',
 'BUREAU_AMT_CREDIT_SUM_SUM',
 'BUREAU_AMT_CREDIT_SUM_MAX',
 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN',
 'BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
 'BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN',
 'BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM',
 'BUREAU_AMT_ANNUITY_MEAN',
 'BUREAU_AMT_ANNUITY_SUM',
 'BUREAU_CREDIT_ACTIVE_Active',
 'BUREAU_CREDIT_ACTIVE_Bad debt',
 'BUREAU_CREDIT_ACTIVE_Closed',
 'BUREAU_CREDIT_ACTIVE_Sold',
 'BUREAU_CREDIT_ACTIVE_nan',
 'BUREAU_CREDIT_TYPE_Another type of loan',
 'BUREAU_CREDIT_TYPE_Car loan',
 'BUREAU_CREDIT_TYPE_Cash loan (non-earmarked)',
 'BUREAU_CREDIT_TYPE_Consumer credit',
 'BUREAU_CREDIT_TYPE_Credit card',
 'BUREAU_CREDIT_TYPE_Interbank credit',
 'BUREAU_CREDIT_TYPE_Loan for business developme

### 8.3 Bureau ratio features

На основе агрегированных `bureau` признаков создадим несколько дополнительных ratio-признаков.

Они помогут описать:

- долю активных кредитов среди всех кредитов клиента;
- долю закрытых кредитов;
- отношение общей задолженности к общей сумме кредитов;
- отношение просроченной суммы к общей сумме кредитов.

In [52]:
bureau_features["BUREAU_ACTIVE_CREDIT_RATIO"] = (
    bureau_features["BUREAU_CREDIT_ACTIVE_Active"] 
    / bureau_features["BUREAU_RECORDS_COUNT"]
)

bureau_features["BUREAU_CLOSED_CREDIT_RATIO"] = (
    bureau_features["BUREAU_CREDIT_ACTIVE_Closed"] 
    / bureau_features["BUREAU_RECORDS_COUNT"]
)

bureau_features["BUREAU_DEBT_TO_CREDIT_RATIO"] = (
    bureau_features["BUREAU_AMT_CREDIT_SUM_DEBT_SUM"] 
    / bureau_features["BUREAU_AMT_CREDIT_SUM_SUM"]
)

bureau_features["BUREAU_OVERDUE_TO_CREDIT_RATIO"] = (
    bureau_features["BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM"] 
    / bureau_features["BUREAU_AMT_CREDIT_SUM_SUM"]
)

bureau_ratio_features = [
    "BUREAU_ACTIVE_CREDIT_RATIO",
    "BUREAU_CLOSED_CREDIT_RATIO",
    "BUREAU_DEBT_TO_CREDIT_RATIO",
    "BUREAU_OVERDUE_TO_CREDIT_RATIO",
]

bureau_features[bureau_ratio_features].describe()

/Users/artem/miniconda3/lib/python3.12/site-packages/numpy/_core/_methods.py:51: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,BUREAU_ACTIVE_CREDIT_RATIO,BUREAU_CLOSED_CREDIT_RATIO,BUREAU_DEBT_TO_CREDIT_RATIO,BUREAU_OVERDUE_TO_CREDIT_RATIO
count,305811.0000,305811.0000,304600.0000,304536.0000
mean,0.4102,0.5860,NaN,inf
std,0.3072,0.3075,NaN,NaN
min,0.0000,0.0000,-inf,0.0000
25%,0.2000,0.4000,0.0000,0.0000
50%,0.3750,0.6250,0.2109,0.0000
75%,0.6000,0.8000,0.4819,0.0000
max,1.0000,1.0000,inf,inf


In [53]:
np.isinf(bureau_features[bureau_ratio_features]).sum()

BUREAU_ACTIVE_CREDIT_RATIO         0
BUREAU_CLOSED_CREDIT_RATIO         0
BUREAU_DEBT_TO_CREDIT_RATIO       65
BUREAU_OVERDUE_TO_CREDIT_RATIO     1
dtype: int64

In [54]:
bureau_features[bureau_ratio_features] = bureau_features[bureau_ratio_features].replace(
    [np.inf, -np.inf],
    np.nan
)
bureau_features.shape

(305811, 43)

### 8.4 Merge bureau features with application

Теперь объединим агрегированные признаки из `bureau` с основной таблицей `application`.

Так как `bureau_features` уже агрегирована до уровня одного клиента, merge можно делать по `SK_ID_CURR`.

Используем `left join`, чтобы сохранить всех клиентов из train/test.

In [55]:
application_shape_before_bureau = application.shape

application = application.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left"
)

application_shape_before_bureau, application.shape

((356255, 147), (356255, 189))

In [56]:
bureau_feature_cols = [
    col for col in bureau_features.columns
    if col != "SK_ID_CURR"
]

application[bureau_feature_cols].isna().all(axis=1).mean()

np.float64(0.14159520568132378)

In [57]:
application[
    [
        "SK_ID_CURR",
        "DATASET",
        "BUREAU_RECORDS_COUNT",
        "BUREAU_ACTIVE_CREDIT_RATIO",
        "BUREAU_CLOSED_CREDIT_RATIO",
        "BUREAU_DEBT_TO_CREDIT_RATIO",
    ]
].head()

,SK_ID_CURR,DATASET,BUREAU_RECORDS_COUNT,BUREAU_ACTIVE_CREDIT_RATIO,BUREAU_CLOSED_CREDIT_RATIO,BUREAU_DEBT_TO_CREDIT_RATIO
0,100002,train,8.0000,0.2500,0.7500,0.2841
1,100003,train,4.0000,0.2500,0.7500,0.0000
2,100004,train,2.0000,0.0000,1.0000,0.0000
3,100006,train,NaN,NaN,NaN,NaN
4,100007,train,1.0000,0.0000,1.0000,0.0000


Вывод:

Признаки из `bureau` успешно агрегированы до уровня клиента и объединены с основной таблицей.

Для клиентов без записей в `bureau` новые признаки будут пропущены. Это нормально: отсутствие кредитной истории в бюро само по себе может быть важным сигналом и позже может быть обработано отдельным флагом или заполнением пропусков.

## 9. Bureau balance features

Таблица `bureau_balance.csv` содержит помесячную историю статусов по кредитам из `bureau`.

Связь идёт через `SK_ID_BUREAU`.

Так как в `bureau_balance` может быть много строк на один прошлый кредит, сначала агрегируем данные до уровня `SK_ID_BUREAU`, затем присоединим их к `bureau`, а потом агрегируем до уровня клиента `SK_ID_CURR`.

In [58]:
bureau_balance = pd.read_csv(RAW_DATA_DIR / "bureau_balance.csv")

bureau_balance.shape

(27299925, 3)

In [59]:
bureau_balance.head()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [60]:
bureau_balance_records_per_credit = (
    bureau_balance.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"]
    .count()
    .sort_values(ascending=False)
)

bureau_balance_records_per_credit.describe()

count   817395.0000
mean        33.3987
std         25.7947
min          1.0000
25%         13.0000
50%         26.0000
75%         48.0000
max         97.0000
Name: MONTHS_BALANCE, dtype: float64

### 9.1 Bureau balance aggregations

Агрегируем `bureau_balance` до уровня одного кредита `SK_ID_BUREAU`.

Создадим признаки:

- количество месяцев истории по кредиту;
- минимальный и максимальный месяц;
- one-hot признаки по статусам;
- количество месяцев в каждом статусе.

После этого присоединим агрегаты к `bureau`, чтобы перейти от уровня кредита к уровню клиента.

In [61]:
bureau_balance_status = pd.get_dummies(
    bureau_balance[["SK_ID_BUREAU", "STATUS"]],
    columns=["STATUS"],
    dummy_na=True
)

bureau_balance_status.head()

,SK_ID_BUREAU,STATUS_0,STATUS_1,STATUS_2,STATUS_3,STATUS_4,STATUS_5,STATUS_C,STATUS_X,STATUS_nan
0,5715448,False,False,False,False,False,False,True,False,False
1,5715448,False,False,False,False,False,False,True,False,False
2,5715448,False,False,False,False,False,False,True,False,False
3,5715448,False,False,False,False,False,False,True,False,False
4,5715448,False,False,False,False,False,False,True,False,False


In [62]:
bureau_balance_agg = bureau_balance.groupby("SK_ID_BUREAU").agg(
    BUREAU_BALANCE_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
    BUREAU_BALANCE_MONTHS_MIN=("MONTHS_BALANCE", "min"),
    BUREAU_BALANCE_MONTHS_MAX=("MONTHS_BALANCE", "max"),
).reset_index()

bureau_balance_status_agg = (
    bureau_balance_status
    .groupby("SK_ID_BUREAU")
    .sum()
    .reset_index()
)

bureau_balance_status_agg = bureau_balance_status_agg.rename(
    columns={
        col: f"BUREAU_BALANCE_{col}"
        for col in bureau_balance_status_agg.columns
        if col != "SK_ID_BUREAU"
    }
)

bureau_balance_agg = bureau_balance_agg.merge(
    bureau_balance_status_agg,
    on="SK_ID_BUREAU",
    how="left"
)

bureau_balance_agg.shape

(817395, 13)

In [63]:
bureau_balance_agg.head()

,SK_ID_BUREAU,BUREAU_BALANCE_MONTHS_COUNT,BUREAU_BALANCE_MONTHS_MIN,BUREAU_BALANCE_MONTHS_MAX,BUREAU_BALANCE_STATUS_0,BUREAU_BALANCE_STATUS_1,BUREAU_BALANCE_STATUS_2,BUREAU_BALANCE_STATUS_3,BUREAU_BALANCE_STATUS_4,BUREAU_BALANCE_STATUS_5,BUREAU_BALANCE_STATUS_C,BUREAU_BALANCE_STATUS_X,BUREAU_BALANCE_STATUS_nan
0,5001709,97,-96,0,0,0,0,0,0,0,86,11,0
1,5001710,83,-82,0,5,0,0,0,0,0,48,30,0
2,5001711,4,-3,0,3,0,0,0,0,0,0,1,0
3,5001712,19,-18,0,10,0,0,0,0,0,9,0,0
4,5001713,22,-21,0,0,0,0,0,0,0,0,22,0


In [64]:
bureau_with_balance = bureau[["SK_ID_CURR", "SK_ID_BUREAU"]].merge(
    bureau_balance_agg,
    on="SK_ID_BUREAU",
    how="left"
)

bureau_with_balance.shape

(1716428, 14)

In [65]:
bureau_balance_client_features = bureau_with_balance.groupby("SK_ID_CURR").agg(
    BUREAU_BALANCE_MONTHS_COUNT_MEAN=("BUREAU_BALANCE_MONTHS_COUNT", "mean"),
    BUREAU_BALANCE_MONTHS_COUNT_SUM=("BUREAU_BALANCE_MONTHS_COUNT", "sum"),
    BUREAU_BALANCE_MONTHS_MIN=("BUREAU_BALANCE_MONTHS_MIN", "min"),
    BUREAU_BALANCE_MONTHS_MAX=("BUREAU_BALANCE_MONTHS_MAX", "max"),
).reset_index()

status_cols = [
    col for col in bureau_with_balance.columns
    if col.startswith("BUREAU_BALANCE_STATUS_")
]

bureau_balance_status_client = (
    bureau_with_balance[["SK_ID_CURR"] + status_cols]
    .groupby("SK_ID_CURR")
    .sum()
    .reset_index()
)

bureau_balance_client_features = bureau_balance_client_features.merge(
    bureau_balance_status_client,
    on="SK_ID_CURR",
    how="left"
)

bureau_balance_client_features.shape

(305811, 14)

In [66]:
bureau_balance_client_features.head()

,SK_ID_CURR,BUREAU_BALANCE_MONTHS_COUNT_MEAN,BUREAU_BALANCE_MONTHS_COUNT_SUM,BUREAU_BALANCE_MONTHS_MIN,BUREAU_BALANCE_MONTHS_MAX,BUREAU_BALANCE_STATUS_0,BUREAU_BALANCE_STATUS_1,BUREAU_BALANCE_STATUS_2,BUREAU_BALANCE_STATUS_3,BUREAU_BALANCE_STATUS_4,BUREAU_BALANCE_STATUS_5,BUREAU_BALANCE_STATUS_C,BUREAU_BALANCE_STATUS_X,BUREAU_BALANCE_STATUS_nan
0,100001,24.5714,172.0000,-51.0000,0.0000,31.0000,1.0000,0.0000,0.0000,0.0000,0.0000,110.0000,30.0000,0.0000
1,100002,13.7500,110.0000,-47.0000,0.0000,45.0000,27.0000,0.0000,0.0000,0.0000,0.0000,23.0000,15.0000,0.0000
2,100003,NaN,0.0000,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,100004,NaN,0.0000,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,100005,7.0000,21.0000,-12.0000,0.0000,14.0000,0.0000,0.0000,0.0000,0.0000,0.0000,5.0000,2.0000,0.0000


### 9.2 Merge bureau_balance features with application

Теперь присоединим признаки из `bureau_balance` к основной таблице `application`.

Используем `left join`, чтобы сохранить всех клиентов из train/test.

In [67]:
application_shape_before_bureau_balance = application.shape

application = application.merge(
    bureau_balance_client_features,
    on="SK_ID_CURR",
    how="left"
)

application_shape_before_bureau_balance, application.shape

((356255, 189), (356255, 202))

In [68]:
bureau_balance_feature_cols = [
    col for col in bureau_balance_client_features.columns
    if col != "SK_ID_CURR"
]

application[bureau_balance_feature_cols].isna().all(axis=1).mean()

np.float64(0.14159520568132378)

In [69]:
application[
    [
        "SK_ID_CURR",
        "DATASET",
        "BUREAU_BALANCE_MONTHS_COUNT_MEAN",
        "BUREAU_BALANCE_MONTHS_COUNT_SUM",
        "BUREAU_BALANCE_STATUS_C",
        "BUREAU_BALANCE_STATUS_X",
    ]
].head()

,SK_ID_CURR,DATASET,BUREAU_BALANCE_MONTHS_COUNT_MEAN,BUREAU_BALANCE_MONTHS_COUNT_SUM,BUREAU_BALANCE_STATUS_C,BUREAU_BALANCE_STATUS_X
0,100002,train,13.7500,110.0000,23.0000,15.0000
1,100003,train,NaN,0.0000,0.0000,0.0000
2,100004,train,NaN,0.0000,0.0000,0.0000
3,100006,train,NaN,NaN,NaN,NaN
4,100007,train,NaN,0.0000,0.0000,0.0000


Вывод:

Признаки из `bureau_balance` успешно агрегированы до уровня клиента и объединены с основной таблицей.

Эти признаки описывают историю статусов прошлых кредитов клиента по месяцам: сколько месяцев есть в истории, сколько было закрытых/активных/просроченных статусов и т.д.

## 10. Previous application features

Таблица `previous_application.csv` содержит информацию о прошлых заявках клиента в Home Credit.

Связь с основной таблицей идёт по `SK_ID_CURR`.

Для одного клиента может быть несколько прошлых заявок, поэтому данные нужно агрегировать до уровня клиента.

Из этой таблицы можно получить признаки:

- количество прошлых заявок;
- средние суммы прошлых кредитов;
- средние аннуитеты;
- долю одобренных и отказанных заявок;
- типы прошлых заявок;
- каналы продаж и продуктовые признаки.

In [70]:
previous_application = pd.read_csv(RAW_DATA_DIR / "previous_application.csv")

previous_application.shape

(1670214, 37)

In [71]:
previous_application.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.4300,17145.0000,17145.0000,0.0000,17145.0000,SATURDAY,15,Y,1,0.0000,0.1828,0.8673,XAP,Approved,-73,Cash through the bank,XAP,NaN,Repeater,Mobile,POS,XNA,Country-wide,35,Connectivity,12.0000,middle,POS mobile with interest,365243.0000,-42.0000,300.0000,-42.0000,-37.0000,0.0000
1,2802425,108129,Cash loans,25188.6150,607500.0000,679671.0000,NaN,607500.0000,THURSDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-164,XNA,XAP,Unaccompanied,Repeater,XNA,Cash,x-sell,Contact center,-1,XNA,36.0000,low_action,Cash X-Sell: low,365243.0000,-134.0000,916.0000,365243.0000,365243.0000,1.0000
2,2523466,122040,Cash loans,15060.7350,112500.0000,136444.5000,NaN,112500.0000,TUESDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-301,Cash through the bank,XAP,"Spouse, partner",Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0000,high,Cash X-Sell: high,365243.0000,-271.0000,59.0000,365243.0000,365243.0000,1.0000
3,2819243,176158,Cash loans,47041.3350,450000.0000,470790.0000,NaN,450000.0000,MONDAY,7,Y,1,NaN,NaN,NaN,XNA,Approved,-512,Cash through the bank,XAP,NaN,Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0000,middle,Cash X-Sell: middle,365243.0000,-482.0000,-152.0000,-182.0000,-177.0000,1.0000
4,1784265,202054,Cash loans,31924.3950,337500.0000,404055.0000,NaN,337500.0000,THURSDAY,9,Y,1,NaN,NaN,NaN,Repairs,Refused,-781,Cash through the bank,HC,NaN,Repeater,XNA,Cash,walk-in,Credit and cash offices,-1,XNA,24.0000,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [72]:
previous_records_per_client = (
    previous_application.groupby("SK_ID_CURR")["SK_ID_PREV"]
    .count()
    .sort_values(ascending=False)
)

previous_records_per_client.describe()

count   338857.0000
mean         4.9290
std          4.2207
min          1.0000
25%          2.0000
50%          4.0000
75%          7.0000
max         77.0000
Name: SK_ID_PREV, dtype: float64

### 10.1 Previous application numeric aggregations

Создадим числовые агрегаты по прошлым заявкам клиента.

Для каждого клиента посчитаем:

- количество прошлых заявок;
- среднюю/максимальную/суммарную сумму прошлых кредитов;
- средний аннуитет;
- среднюю сумму заявки;
- среднюю стоимость товара;
- среднюю продолжительность решения по заявке;
- среднее количество платежей.

In [73]:
previous_agg = previous_application.groupby("SK_ID_CURR").agg(
    PREV_APPLICATION_COUNT=("SK_ID_PREV", "count"),
    PREV_AMT_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    PREV_AMT_APPLICATION_MEAN=("AMT_APPLICATION", "mean"),
    PREV_AMT_APPLICATION_MAX=("AMT_APPLICATION", "max"),
    PREV_AMT_APPLICATION_SUM=("AMT_APPLICATION", "sum"),
    PREV_AMT_CREDIT_MEAN=("AMT_CREDIT", "mean"),
    PREV_AMT_CREDIT_MAX=("AMT_CREDIT", "max"),
    PREV_AMT_CREDIT_SUM=("AMT_CREDIT", "sum"),
    PREV_AMT_DOWN_PAYMENT_MEAN=("AMT_DOWN_PAYMENT", "mean"),
    PREV_AMT_GOODS_PRICE_MEAN=("AMT_GOODS_PRICE", "mean"),
    PREV_HOUR_APPR_PROCESS_START_MEAN=("HOUR_APPR_PROCESS_START", "mean"),
    PREV_RATE_DOWN_PAYMENT_MEAN=("RATE_DOWN_PAYMENT", "mean"),
    PREV_DAYS_DECISION_MEAN=("DAYS_DECISION", "mean"),
    PREV_DAYS_DECISION_MIN=("DAYS_DECISION", "min"),
    PREV_DAYS_DECISION_MAX=("DAYS_DECISION", "max"),
    PREV_CNT_PAYMENT_MEAN=("CNT_PAYMENT", "mean"),
    PREV_CNT_PAYMENT_MAX=("CNT_PAYMENT", "max"),
).reset_index()

previous_agg.shape

(338857, 18)

In [74]:
previous_agg.head()

,SK_ID_CURR,PREV_APPLICATION_COUNT,PREV_AMT_ANNUITY_MEAN,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_APPLICATION_SUM,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_CREDIT_SUM,PREV_AMT_DOWN_PAYMENT_MEAN,PREV_AMT_GOODS_PRICE_MEAN,PREV_HOUR_APPR_PROCESS_START_MEAN,PREV_RATE_DOWN_PAYMENT_MEAN,PREV_DAYS_DECISION_MEAN,PREV_DAYS_DECISION_MIN,PREV_DAYS_DECISION_MAX,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX
0,100001,1,3951.0000,24835.5000,24835.5000,24835.5000,23787.0000,23787.0000,23787.0000,2520.0000,24835.5000,13.0000,0.1043,-1740.0000,-1740,-1740,8.0000,8.0000
1,100002,1,9251.7750,179055.0000,179055.0000,179055.0000,179055.0000,179055.0000,179055.0000,0.0000,179055.0000,9.0000,0.0000,-606.0000,-606,-606,24.0000,24.0000
2,100003,3,56553.9900,435436.5000,900000.0000,1306309.5000,484191.0000,1035882.0000,1452573.0000,3442.5000,435436.5000,14.6667,0.0500,-1305.0000,-2341,-746,10.0000,12.0000
3,100004,1,5357.2500,24282.0000,24282.0000,24282.0000,20106.0000,20106.0000,20106.0000,4860.0000,24282.0000,5.0000,0.2120,-815.0000,-815,-815,4.0000,4.0000
4,100005,2,4813.2000,22308.7500,44617.5000,44617.5000,20076.7500,40153.5000,40153.5000,4464.0000,44617.5000,10.5000,0.1090,-536.0000,-757,-315,12.0000,12.0000


In [75]:
previous_agg.isna().mean().sort_values(ascending=False).head(10)

PREV_AMT_DOWN_PAYMENT_MEAN    0.0593
PREV_RATE_DOWN_PAYMENT_MEAN   0.0593
PREV_AMT_GOODS_PRICE_MEAN     0.0031
PREV_AMT_ANNUITY_MEAN         0.0014
PREV_CNT_PAYMENT_MEAN         0.0014
PREV_CNT_PAYMENT_MAX          0.0014
PREV_AMT_CREDIT_MEAN          0.0000
PREV_AMT_CREDIT_MAX           0.0000
PREV_AMT_CREDIT_SUM           0.0000
PREV_APPLICATION_COUNT        0.0000
dtype: float64

In [76]:
previous_agg["PREV_CREDIT_TO_APPLICATION_RATIO"] = (
    previous_agg["PREV_AMT_CREDIT_SUM"] / previous_agg["PREV_AMT_APPLICATION_SUM"]
)

previous_ratio_features = [
    "PREV_CREDIT_TO_APPLICATION_RATIO",
]

np.isinf(previous_agg[previous_ratio_features]).sum()

PREV_CREDIT_TO_APPLICATION_RATIO    853
dtype: int64

In [77]:
previous_agg[previous_ratio_features] = previous_agg[previous_ratio_features].replace(
    [np.inf, -np.inf],
    np.nan
)

### 10.2 Previous application categorical aggregations

Теперь создадим признаки на основе категориальных колонок `previous_application`.

Особенно важны:

- `NAME_CONTRACT_STATUS` — статус прошлой заявки;
- `NAME_CONTRACT_TYPE` — тип договора;
- `NAME_CLIENT_TYPE` — тип клиента;
- `CHANNEL_TYPE` — канал привлечения;
- `NAME_YIELD_GROUP` — группа доходности продукта.

После one-hot encoding агрегируем признаки по клиенту.

In [78]:
previous_cat_cols = [
    "NAME_CONTRACT_STATUS",
    "NAME_CONTRACT_TYPE",
    "NAME_CLIENT_TYPE",
    "CHANNEL_TYPE",
    "NAME_YIELD_GROUP",
]

previous_cat = pd.get_dummies(
    previous_application[["SK_ID_CURR"] + previous_cat_cols],
    columns=previous_cat_cols,
    dummy_na=True
)

previous_cat.head()

,SK_ID_CURR,NAME_CONTRACT_STATUS_Approved,NAME_CONTRACT_STATUS_Canceled,NAME_CONTRACT_STATUS_Refused,NAME_CONTRACT_STATUS_Unused offer,NAME_CONTRACT_STATUS_nan,NAME_CONTRACT_TYPE_Cash loans,NAME_CONTRACT_TYPE_Consumer loans,NAME_CONTRACT_TYPE_Revolving loans,NAME_CONTRACT_TYPE_XNA,NAME_CONTRACT_TYPE_nan,NAME_CLIENT_TYPE_New,NAME_CLIENT_TYPE_Refreshed,NAME_CLIENT_TYPE_Repeater,NAME_CLIENT_TYPE_XNA,NAME_CLIENT_TYPE_nan,CHANNEL_TYPE_AP+ (Cash loan),CHANNEL_TYPE_Car dealer,CHANNEL_TYPE_Channel of corporate sales,CHANNEL_TYPE_Contact center,CHANNEL_TYPE_Country-wide,CHANNEL_TYPE_Credit and cash offices,CHANNEL_TYPE_Regional / Local,CHANNEL_TYPE_Stone,CHANNEL_TYPE_nan,NAME_YIELD_GROUP_XNA,NAME_YIELD_GROUP_high,NAME_YIELD_GROUP_low_action,NAME_YIELD_GROUP_low_normal,NAME_YIELD_GROUP_middle,NAME_YIELD_GROUP_nan
0,271877,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False
1,108129,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False
2,122040,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False
3,176158,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False
4,202054,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False


In [79]:
previous_cat_agg = previous_cat.groupby("SK_ID_CURR").sum().reset_index()

previous_cat_agg = previous_cat_agg.rename(
    columns={
        col: f"PREV_{col}"
        for col in previous_cat_agg.columns
        if col != "SK_ID_CURR"
    }
)

previous_cat_agg.shape

(338857, 31)

In [80]:
previous_features = previous_agg.merge(
    previous_cat_agg,
    on="SK_ID_CURR",
    how="left"
)

previous_features.shape

(338857, 49)

In [81]:
previous_features.head()

,SK_ID_CURR,PREV_APPLICATION_COUNT,PREV_AMT_ANNUITY_MEAN,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_APPLICATION_SUM,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_CREDIT_SUM,PREV_AMT_DOWN_PAYMENT_MEAN,PREV_AMT_GOODS_PRICE_MEAN,PREV_HOUR_APPR_PROCESS_START_MEAN,PREV_RATE_DOWN_PAYMENT_MEAN,PREV_DAYS_DECISION_MEAN,PREV_DAYS_DECISION_MIN,PREV_DAYS_DECISION_MAX,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX,PREV_CREDIT_TO_APPLICATION_RATIO,PREV_NAME_CONTRACT_STATUS_Approved,PREV_NAME_CONTRACT_STATUS_Canceled,PREV_NAME_CONTRACT_STATUS_Refused,PREV_NAME_CONTRACT_STATUS_Unused offer,PREV_NAME_CONTRACT_STATUS_nan,PREV_NAME_CONTRACT_TYPE_Cash loans,PREV_NAME_CONTRACT_TYPE_Consumer loans,PREV_NAME_CONTRACT_TYPE_Revolving loans,PREV_NAME_CONTRACT_TYPE_XNA,PREV_NAME_CONTRACT_TYPE_nan,PREV_NAME_CLIENT_TYPE_New,PREV_NAME_CLIENT_TYPE_Refreshed,PREV_NAME_CLIENT_TYPE_Repeater,PREV_NAME_CLIENT_TYPE_XNA,PREV_NAME_CLIENT_TYPE_nan,PREV_CHANNEL_TYPE_AP+ (Cash loan),PREV_CHANNEL_TYPE_Car dealer,PREV_CHANNEL_TYPE_Channel of corporate sales,PREV_CHANNEL_TYPE_Contact center,PREV_CHANNEL_TYPE_Country-wide,PREV_CHANNEL_TYPE_Credit and cash offices,PREV_CHANNEL_TYPE_Regional / Local,PREV_CHANNEL_TYPE_Stone,PREV_CHANNEL_TYPE_nan,PREV_NAME_YIELD_GROUP_XNA,PREV_NAME_YIELD_GROUP_high,PREV_NAME_YIELD_GROUP_low_action,PREV_NAME_YIELD_GROUP_low_normal,PREV_NAME_YIELD_GROUP_middle,PREV_NAME_YIELD_GROUP_nan
0,100001,1,3951.0000,24835.5000,24835.5000,24835.5000,23787.0000,23787.0000,23787.0000,2520.0000,24835.5000,13.0000,0.1043,-1740.0000,-1740,-1740,8.0000,8.0000,0.9578,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
1,100002,1,9251.7750,179055.0000,179055.0000,179055.0000,179055.0000,179055.0000,179055.0000,0.0000,179055.0000,9.0000,0.0000,-606.0000,-606,-606,24.0000,24.0000,1.0000,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0
2,100003,3,56553.9900,435436.5000,900000.0000,1306309.5000,484191.0000,1035882.0000,1452573.0000,3442.5000,435436.5000,14.6667,0.0500,-1305.0000,-2341,-746,10.0000,12.0000,1.1120,3,0,0,0,0,1,2,0,0,0,0,2,1,0,0,0,0,0,0,1,1,0,1,0,0,0,0,1,2,0
3,100004,1,5357.2500,24282.0000,24282.0000,24282.0000,20106.0000,20106.0000,20106.0000,4860.0000,24282.0000,5.0000,0.2120,-815.0000,-815,-815,4.0000,4.0000,0.8280,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
4,100005,2,4813.2000,22308.7500,44617.5000,44617.5000,20076.7500,40153.5000,40153.5000,4464.0000,44617.5000,10.5000,0.1090,-536.0000,-757,-315,12.0000,12.0000,0.8999,1,1,0,0,0,1,1,0,0,0,1,0,1,0,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0,0


In [82]:
previous_features["PREV_APPROVED_RATIO"] = (
    previous_features["PREV_NAME_CONTRACT_STATUS_Approved"]
    / previous_features["PREV_APPLICATION_COUNT"]
)

previous_features["PREV_REFUSED_RATIO"] = (
    previous_features["PREV_NAME_CONTRACT_STATUS_Refused"]
    / previous_features["PREV_APPLICATION_COUNT"]
)

previous_features["PREV_CANCELED_RATIO"] = (
    previous_features["PREV_NAME_CONTRACT_STATUS_Canceled"]
    / previous_features["PREV_APPLICATION_COUNT"]
)

previous_status_ratio_features = [
    "PREV_APPROVED_RATIO",
    "PREV_REFUSED_RATIO",
    "PREV_CANCELED_RATIO",
]

previous_features[previous_status_ratio_features].describe()

,PREV_APPROVED_RATIO,PREV_REFUSED_RATIO,PREV_CANCELED_RATIO
count,338857.0000,338857.0000,338857.0000
mean,0.7445,0.1114,0.1292
std,0.2632,0.1837,0.1887
min,0.0000,0.0000,0.0000
25%,0.5000,0.0000,0.0000
50%,0.7778,0.0000,0.0000
75%,1.0000,0.2000,0.2500
max,1.0000,1.0000,1.0000


In [83]:
np.isinf(previous_features[previous_status_ratio_features]).sum()

PREV_APPROVED_RATIO    0
PREV_REFUSED_RATIO     0
PREV_CANCELED_RATIO    0
dtype: int64

### 10.3 Merge previous application features with application

Теперь присоединим агрегированные признаки из `previous_application` к основной таблице.

Используем `left join`, чтобы сохранить всех клиентов из train/test.

In [84]:
application_shape_before_previous = application.shape

application = application.merge(
    previous_features,
    on="SK_ID_CURR",
    how="left"
)

application_shape_before_previous, application.shape

((356255, 202), (356255, 253))

In [85]:
previous_feature_cols = [
    col for col in previous_features.columns
    if col != "SK_ID_CURR"
]

application[previous_feature_cols].isna().all(axis=1).mean()

np.float64(0.04883580581325175)

In [86]:
application[
    [
        "SK_ID_CURR",
        "DATASET",
        "PREV_APPLICATION_COUNT",
        "PREV_APPROVED_RATIO",
        "PREV_REFUSED_RATIO",
        "PREV_CANCELED_RATIO",
        "PREV_CREDIT_TO_APPLICATION_RATIO",
    ]
].head()

,SK_ID_CURR,DATASET,PREV_APPLICATION_COUNT,PREV_APPROVED_RATIO,PREV_REFUSED_RATIO,PREV_CANCELED_RATIO,PREV_CREDIT_TO_APPLICATION_RATIO
0,100002,train,1.0000,1.0000,0.0000,0.0000,1.0000
1,100003,train,3.0000,1.0000,0.0000,0.0000,1.1120
2,100004,train,1.0000,1.0000,0.0000,0.0000,0.8280
3,100006,train,9.0000,0.5556,0.1111,0.3333,1.0716
4,100007,train,6.0000,1.0000,0.0000,0.0000,1.1070


Вывод:

Признаки из `previous_application` успешно агрегированы до уровня клиента и объединены с основной таблицей.

Эти признаки описывают историю прошлых заявок клиента в Home Credit: количество заявок, суммы кредитов, статусы заявок, долю одобренных и отказанных заявок.

## 11. POS_CASH_balance features

Таблица `POS_CASH_balance.csv` содержит помесячную историю POS-кредитов и cash-кредитов клиента.

Связь идёт через:

- `SK_ID_PREV` — предыдущая заявка;
- `SK_ID_CURR` — клиент.

Так как на одного клиента может быть много записей, агрегируем таблицу до уровня `SK_ID_CURR`.

In [87]:
pos_cash = pd.read_csv(RAW_DATA_DIR / "POS_CASH_balance.csv")

pos_cash.shape

(10001358, 8)

In [88]:
pos_cash.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0000,45.0000,Active,0,0
1,1715348,367990,-33,36.0000,35.0000,Active,0,0
2,1784872,397406,-32,12.0000,9.0000,Active,0,0
3,1903291,269225,-35,48.0000,42.0000,Active,0,0
4,2341044,334279,-35,36.0000,35.0000,Active,0,0


In [89]:
pos_cash_records_per_client = (
    pos_cash.groupby("SK_ID_CURR")["SK_ID_PREV"]
    .count()
    .sort_values(ascending=False)
)

pos_cash_records_per_client.describe()

count   337252.0000
mean        29.6554
std         24.5320
min          1.0000
25%         12.0000
50%         22.0000
75%         39.0000
max        295.0000
Name: SK_ID_PREV, dtype: float64

### 11.1 POS_CASH aggregations

Создадим агрегированные признаки по POS/CASH истории клиента:

- количество записей;
- количество уникальных прошлых заявок;
- средний и максимальный номер месяца;
- средний и максимальный срок до конца кредита;
- среднее и максимальное количество просрочек;
- статусы договоров через one-hot encoding.

In [90]:
pos_cash_agg = pos_cash.groupby("SK_ID_CURR").agg(
    POS_RECORDS_COUNT=("SK_ID_PREV", "count"),
    POS_PREV_UNIQUE_COUNT=("SK_ID_PREV", "nunique"),
    POS_MONTHS_BALANCE_MEAN=("MONTHS_BALANCE", "mean"),
    POS_MONTHS_BALANCE_MIN=("MONTHS_BALANCE", "min"),
    POS_MONTHS_BALANCE_MAX=("MONTHS_BALANCE", "max"),
    POS_CNT_INSTALMENT_MEAN=("CNT_INSTALMENT", "mean"),
    POS_CNT_INSTALMENT_MAX=("CNT_INSTALMENT", "max"),
    POS_CNT_INSTALMENT_FUTURE_MEAN=("CNT_INSTALMENT_FUTURE", "mean"),
    POS_CNT_INSTALMENT_FUTURE_MAX=("CNT_INSTALMENT_FUTURE", "max"),
    POS_SK_DPD_MEAN=("SK_DPD", "mean"),
    POS_SK_DPD_MAX=("SK_DPD", "max"),
    POS_SK_DPD_DEF_MEAN=("SK_DPD_DEF", "mean"),
    POS_SK_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
).reset_index()

pos_cash_agg.shape

(337252, 14)

In [91]:
pos_cash_cat = pd.get_dummies(
    pos_cash[["SK_ID_CURR", "NAME_CONTRACT_STATUS"]],
    columns=["NAME_CONTRACT_STATUS"],
    dummy_na=True
)

pos_cash_cat_agg = pos_cash_cat.groupby("SK_ID_CURR").sum().reset_index()

pos_cash_cat_agg = pos_cash_cat_agg.rename(
    columns={
        col: f"POS_{col}"
        for col in pos_cash_cat_agg.columns
        if col != "SK_ID_CURR"
    }
)

pos_cash_features = pos_cash_agg.merge(
    pos_cash_cat_agg,
    on="SK_ID_CURR",
    how="left"
)

pos_cash_features.shape

(337252, 24)

In [92]:
pos_cash_features.head()

,SK_ID_CURR,POS_RECORDS_COUNT,POS_PREV_UNIQUE_COUNT,POS_MONTHS_BALANCE_MEAN,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX,POS_NAME_CONTRACT_STATUS_Active,POS_NAME_CONTRACT_STATUS_Amortized debt,POS_NAME_CONTRACT_STATUS_Approved,POS_NAME_CONTRACT_STATUS_Canceled,POS_NAME_CONTRACT_STATUS_Completed,POS_NAME_CONTRACT_STATUS_Demand,POS_NAME_CONTRACT_STATUS_Returned to the store,POS_NAME_CONTRACT_STATUS_Signed,POS_NAME_CONTRACT_STATUS_XNA,POS_NAME_CONTRACT_STATUS_nan
0,100001,9,2,-72.5556,-96,-53,4.0000,4.0000,1.4444,4.0000,0.7778,7,0.7778,7,7,0,0,0,2,0,0,0,0,0
1,100002,19,1,-10.0000,-19,-1,24.0000,24.0000,15.0000,24.0000,0.0000,0,0.0000,0,19,0,0,0,0,0,0,0,0,0
2,100003,28,3,-43.7857,-77,-18,10.1071,12.0000,5.7857,12.0000,0.0000,0,0.0000,0,26,0,0,0,2,0,0,0,0,0
3,100004,4,1,-25.5000,-27,-24,3.7500,4.0000,2.2500,4.0000,0.0000,0,0.0000,0,3,0,0,0,1,0,0,0,0,0
4,100005,11,1,-20.0000,-25,-15,11.7000,12.0000,7.2000,12.0000,0.0000,0,0.0000,0,9,0,0,0,1,0,0,1,0,0


In [93]:
pos_cash_features.isna().mean().sort_values(ascending=False).head(10)

POS_CNT_INSTALMENT_MEAN                          0.0001
POS_CNT_INSTALMENT_MAX                           0.0001
POS_CNT_INSTALMENT_FUTURE_MEAN                   0.0001
POS_CNT_INSTALMENT_FUTURE_MAX                    0.0001
SK_ID_CURR                                       0.0000
POS_NAME_CONTRACT_STATUS_Active                  0.0000
POS_NAME_CONTRACT_STATUS_XNA                     0.0000
POS_NAME_CONTRACT_STATUS_Signed                  0.0000
POS_NAME_CONTRACT_STATUS_Returned to the store   0.0000
POS_NAME_CONTRACT_STATUS_Demand                  0.0000
dtype: float64

### 11.2 Merge POS_CASH features with application

Теперь присоединим агрегированные признаки из `POS_CASH_balance` к основной таблице.

Эти признаки описывают историю POS/CASH кредитов клиента: количество записей, количество прошлых договоров, сроки, просрочки и статусы договоров.

In [94]:
application_shape_before_pos = application.shape

application = application.merge(
    pos_cash_features,
    on="SK_ID_CURR",
    how="left"
)

application_shape_before_pos, application.shape

((356255, 253), (356255, 276))

In [95]:
pos_cash_feature_cols = [
    col for col in pos_cash_features.columns
    if col != "SK_ID_CURR"
]

application[pos_cash_feature_cols].isna().all(axis=1).mean()

np.float64(0.053341005740270314)

In [96]:
application[
    [
        "SK_ID_CURR",
        "DATASET",
        "POS_RECORDS_COUNT",
        "POS_PREV_UNIQUE_COUNT",
        "POS_SK_DPD_MEAN",
        "POS_SK_DPD_MAX",
    ]
].head()

,SK_ID_CURR,DATASET,POS_RECORDS_COUNT,POS_PREV_UNIQUE_COUNT,POS_SK_DPD_MEAN,POS_SK_DPD_MAX
0,100002,train,19.0000,1.0000,0.0000,0.0000
1,100003,train,28.0000,3.0000,0.0000,0.0000
2,100004,train,4.0000,1.0000,0.0000,0.0000
3,100006,train,21.0000,3.0000,0.0000,0.0000
4,100007,train,66.0000,5.0000,0.0000,0.0000


Вывод:

Признаки из `POS_CASH_balance` успешно агрегированы до уровня клиента и объединены с основной таблицей.

Они добавляют информацию о POS/CASH истории клиента, количестве прошлых договоров, сроках и просрочках.

## 12. Installments payments features

Таблица `installments_payments.csv` содержит информацию о платежах клиента по предыдущим кредитам.

Связь идёт через:

- `SK_ID_PREV` — предыдущая заявка;
- `SK_ID_CURR` — клиент.

Из этой таблицы можно создать важные признаки:

- количество платежей;
- сумма платежей;
- средняя сумма платежа;
- задержка платежа;
- разница между фактическим платежом и ожидаемым;
- доля просроченных платежей.

In [97]:
installments = pd.read_csv(RAW_DATA_DIR / "installments_payments.csv")

installments.shape

(13605401, 8)

In [98]:
installments.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0000,6,-1180.0000,-1187.0000,6948.3600,6948.3600
1,1330831,151639,0.0000,34,-2156.0000,-2156.0000,1716.5250,1716.5250
2,2085231,193053,2.0000,1,-63.0000,-63.0000,25425.0000,25425.0000
3,2452527,199697,1.0000,3,-2418.0000,-2426.0000,24350.1300,24350.1300
4,2714724,167756,1.0000,2,-1383.0000,-1366.0000,2165.0400,2160.5850


In [99]:
installments_records_per_client = (
    installments.groupby("SK_ID_CURR")["SK_ID_PREV"]
    .count()
    .sort_values(ascending=False)
)

installments_records_per_client.describe()

count   339587.0000
mean        40.0646
std         41.0533
min          1.0000
25%         12.0000
50%         25.0000
75%         51.0000
max        372.0000
Name: SK_ID_PREV, dtype: float64

### 12.1 Installments payment features

Создадим признаки на уровне отдельного платежа:

- `INSTALLMENT_PAYMENT_DELAY` — задержка платежа;
- `INSTALLMENT_PAYMENT_DIFF` — разница между фактической и ожидаемой суммой;
- `INSTALLMENT_LATE_PAYMENT_FLAG` — был ли платёж просрочен;
- `INSTALLMENT_UNDERPAYMENT_FLAG` — был ли платёж меньше ожидаемой суммы.

После этого агрегируем признаки до уровня клиента.

In [100]:
installments["INSTALLMENT_PAYMENT_DELAY"] = (
    installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]
)

installments["INSTALLMENT_PAYMENT_DIFF"] = (
    installments["AMT_PAYMENT"] - installments["AMT_INSTALMENT"]
)

installments["INSTALLMENT_LATE_PAYMENT_FLAG"] = (
    installments["INSTALLMENT_PAYMENT_DELAY"] > 0
).astype(int)

installments["INSTALLMENT_UNDERPAYMENT_FLAG"] = (
    installments["INSTALLMENT_PAYMENT_DIFF"] < 0
).astype(int)

installments[
    [
        "SK_ID_CURR",
        "SK_ID_PREV",
        "DAYS_INSTALMENT",
        "DAYS_ENTRY_PAYMENT",
        "INSTALLMENT_PAYMENT_DELAY",
        "AMT_INSTALMENT",
        "AMT_PAYMENT",
        "INSTALLMENT_PAYMENT_DIFF",
        "INSTALLMENT_LATE_PAYMENT_FLAG",
        "INSTALLMENT_UNDERPAYMENT_FLAG",
    ]
].head()

,SK_ID_CURR,SK_ID_PREV,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,INSTALLMENT_PAYMENT_DELAY,AMT_INSTALMENT,AMT_PAYMENT,INSTALLMENT_PAYMENT_DIFF,INSTALLMENT_LATE_PAYMENT_FLAG,INSTALLMENT_UNDERPAYMENT_FLAG
0,161674,1054186,-1180.0000,-1187.0000,-7.0000,6948.3600,6948.3600,0.0000,0,0
1,151639,1330831,-2156.0000,-2156.0000,0.0000,1716.5250,1716.5250,0.0000,0,0
2,193053,2085231,-63.0000,-63.0000,0.0000,25425.0000,25425.0000,0.0000,0,0
3,199697,2452527,-2418.0000,-2426.0000,-8.0000,24350.1300,24350.1300,0.0000,0,0
4,167756,2714724,-1383.0000,-1366.0000,17.0000,2165.0400,2160.5850,-4.4550,1,1


### 12.2 Installments aggregations

Агрегируем платежные признаки до уровня клиента.

Для каждого клиента посчитаем:

- количество платежей;
- количество уникальных прошлых договоров;
- среднюю и максимальную задержку;
- долю просроченных платежей;
- долю неполных платежей;
- среднюю и суммарную сумму платежей;
- среднюю разницу между фактической и ожидаемой суммой.

In [101]:
installments_features = installments.groupby("SK_ID_CURR").agg(
    INSTAL_RECORDS_COUNT=("SK_ID_PREV", "count"),
    INSTAL_PREV_UNIQUE_COUNT=("SK_ID_PREV", "nunique"),
    INSTAL_NUM_INSTALMENT_VERSION_MEAN=("NUM_INSTALMENT_VERSION", "mean"),
    INSTAL_NUM_INSTALMENT_NUMBER_MAX=("NUM_INSTALMENT_NUMBER", "max"),
    INSTAL_DAYS_INSTALMENT_MEAN=("DAYS_INSTALMENT", "mean"),
    INSTAL_DAYS_ENTRY_PAYMENT_MEAN=("DAYS_ENTRY_PAYMENT", "mean"),
    INSTAL_PAYMENT_DELAY_MEAN=("INSTALLMENT_PAYMENT_DELAY", "mean"),
    INSTAL_PAYMENT_DELAY_MAX=("INSTALLMENT_PAYMENT_DELAY", "max"),
    INSTAL_PAYMENT_DELAY_SUM=("INSTALLMENT_PAYMENT_DELAY", "sum"),
    INSTAL_PAYMENT_DIFF_MEAN=("INSTALLMENT_PAYMENT_DIFF", "mean"),
    INSTAL_PAYMENT_DIFF_SUM=("INSTALLMENT_PAYMENT_DIFF", "sum"),
    INSTAL_LATE_PAYMENT_MEAN=("INSTALLMENT_LATE_PAYMENT_FLAG", "mean"),
    INSTAL_LATE_PAYMENT_SUM=("INSTALLMENT_LATE_PAYMENT_FLAG", "sum"),
    INSTAL_UNDERPAYMENT_MEAN=("INSTALLMENT_UNDERPAYMENT_FLAG", "mean"),
    INSTAL_UNDERPAYMENT_SUM=("INSTALLMENT_UNDERPAYMENT_FLAG", "sum"),
    INSTAL_AMT_INSTALMENT_MEAN=("AMT_INSTALMENT", "mean"),
    INSTAL_AMT_INSTALMENT_SUM=("AMT_INSTALMENT", "sum"),
    INSTAL_AMT_PAYMENT_MEAN=("AMT_PAYMENT", "mean"),
    INSTAL_AMT_PAYMENT_SUM=("AMT_PAYMENT", "sum"),
).reset_index()

installments_features.shape

(339587, 20)

In [102]:
installments_features.head()

,SK_ID_CURR,INSTAL_RECORDS_COUNT,INSTAL_PREV_UNIQUE_COUNT,INSTAL_NUM_INSTALMENT_VERSION_MEAN,INSTAL_NUM_INSTALMENT_NUMBER_MAX,INSTAL_DAYS_INSTALMENT_MEAN,INSTAL_DAYS_ENTRY_PAYMENT_MEAN,INSTAL_PAYMENT_DELAY_MEAN,INSTAL_PAYMENT_DELAY_MAX,INSTAL_PAYMENT_DELAY_SUM,INSTAL_PAYMENT_DIFF_MEAN,INSTAL_PAYMENT_DIFF_SUM,INSTAL_LATE_PAYMENT_MEAN,INSTAL_LATE_PAYMENT_SUM,INSTAL_UNDERPAYMENT_MEAN,INSTAL_UNDERPAYMENT_SUM,INSTAL_AMT_INSTALMENT_MEAN,INSTAL_AMT_INSTALMENT_SUM,INSTAL_AMT_PAYMENT_MEAN,INSTAL_AMT_PAYMENT_SUM
0,100001,7,2,1.1429,4,-2187.7143,-2195.0000,-7.2857,11.0000,-51.0000,0.0000,0.0000,0.1429,1,0.0000,0,5885.1321,41195.9250,5885.1321,41195.9250
1,100002,19,1,1.0526,19,-295.0000,-315.4211,-20.4211,-12.0000,-388.0000,0.0000,0.0000,0.0000,0,0.0000,0,11559.2471,219625.6950,11559.2471,219625.6950
2,100003,25,3,1.0400,12,-1378.1600,-1385.3200,-7.1600,-1.0000,-179.0000,0.0000,0.0000,0.0000,0,0.0000,0,64754.5860,1618864.6500,64754.5860,1618864.6500
3,100004,3,1,1.3333,3,-754.0000,-761.6667,-7.6667,-3.0000,-23.0000,0.0000,0.0000,0.0000,0,0.0000,0,7096.1550,21288.4650,7096.1550,21288.4650
4,100005,9,1,1.1111,9,-586.0000,-609.5556,-23.5556,1.0000,-212.0000,0.0000,0.0000,0.1111,1,0.0000,0,6240.2050,56161.8450,6240.2050,56161.8450


In [103]:
installments_features.isna().mean().sort_values(ascending=False).head(10)

INSTAL_PAYMENT_DIFF_MEAN         0.0000
INSTAL_AMT_PAYMENT_MEAN          0.0000
INSTAL_DAYS_ENTRY_PAYMENT_MEAN   0.0000
INSTAL_PAYMENT_DELAY_MEAN        0.0000
INSTAL_PAYMENT_DELAY_MAX         0.0000
INSTAL_LATE_PAYMENT_MEAN         0.0000
INSTAL_AMT_INSTALMENT_SUM        0.0000
INSTAL_AMT_INSTALMENT_MEAN       0.0000
INSTAL_UNDERPAYMENT_SUM          0.0000
INSTAL_UNDERPAYMENT_MEAN         0.0000
dtype: float64

In [104]:
np.isinf(
    installments_features.select_dtypes(include=["int64", "float64"])
).sum().sort_values(ascending=False).head(10)

SK_ID_CURR                    0
INSTAL_RECORDS_COUNT          0
INSTAL_AMT_PAYMENT_MEAN       0
INSTAL_AMT_INSTALMENT_SUM     0
INSTAL_AMT_INSTALMENT_MEAN    0
INSTAL_UNDERPAYMENT_SUM       0
INSTAL_UNDERPAYMENT_MEAN      0
INSTAL_LATE_PAYMENT_SUM       0
INSTAL_LATE_PAYMENT_MEAN      0
INSTAL_PAYMENT_DIFF_SUM       0
dtype: int64

### 12.3 Merge installments features with application

Теперь присоединим агрегированные признаки из `installments_payments` к основной таблице.

Эти признаки описывают платежное поведение клиента: количество платежей, задержки, неполные платежи и суммы оплат.

In [105]:
application_shape_before_installments = application.shape

application = application.merge(
    installments_features,
    on="SK_ID_CURR",
    how="left"
)

application_shape_before_installments, application.shape

((356255, 276), (356255, 295))

In [106]:
installments_feature_cols = [
    col for col in installments_features.columns
    if col != "SK_ID_CURR"
]

application[installments_feature_cols].isna().all(axis=1).mean()

np.float64(0.04678671176544891)

In [107]:
application[
    [
        "SK_ID_CURR",
        "DATASET",
        "INSTAL_RECORDS_COUNT",
        "INSTAL_PREV_UNIQUE_COUNT",
        "INSTAL_PAYMENT_DELAY_MEAN",
        "INSTAL_PAYMENT_DELAY_MAX",
        "INSTAL_LATE_PAYMENT_MEAN",
        "INSTAL_UNDERPAYMENT_MEAN",
    ]
].head()

,SK_ID_CURR,DATASET,INSTAL_RECORDS_COUNT,INSTAL_PREV_UNIQUE_COUNT,INSTAL_PAYMENT_DELAY_MEAN,INSTAL_PAYMENT_DELAY_MAX,INSTAL_LATE_PAYMENT_MEAN,INSTAL_UNDERPAYMENT_MEAN
0,100002,train,19.0000,1.0000,-20.4211,-12.0000,0.0000,0.0000
1,100003,train,25.0000,3.0000,-7.1600,-1.0000,0.0000,0.0000
2,100004,train,3.0000,1.0000,-7.6667,-3.0000,0.0000,0.0000
3,100006,train,16.0000,3.0000,-19.3750,-1.0000,0.0000,0.0000
4,100007,train,66.0000,5.0000,-3.6364,12.0000,0.2424,0.0909


Вывод:

Признаки из `installments_payments` успешно агрегированы до уровня клиента и объединены с основной таблицей.

Это один из важных блоков feature engineering, потому что он описывает реальное платежное поведение клиента: платил ли клиент вовремя, были ли задержки и неполные платежи.

## 13. Credit card balance features

Таблица `credit_card_balance.csv` содержит помесячную историю по кредитным картам клиента.

Связь идёт через:

- `SK_ID_PREV` — предыдущая заявка;
- `SK_ID_CURR` — клиент.

Из этой таблицы можно получить признаки:

- количество записей по кредитным картам;
- количество уникальных кредитных карт;
- средний баланс;
- средний кредитный лимит;
- среднюю задолженность;
- средние просрочки;
- статусы договоров.

In [108]:
credit_card = pd.read_csv(RAW_DATA_DIR / "credit_card_balance.csv")

credit_card.shape

(3840312, 23)

In [109]:
credit_card.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.9700,135000,0.0000,877.5000,0.0000,877.5000,1700.3250,1800.0000,1800.0000,0.0000,0.0000,0.0000,0.0000,1,0.0000,1.0000,35.0000,Active,0,0
1,2582071,363914,-1,63975.5550,45000,2250.0000,2250.0000,0.0000,0.0000,2250.0000,2250.0000,2250.0000,60175.0800,64875.5550,64875.5550,1.0000,1,0.0000,0.0000,69.0000,Active,0,0
2,1740877,371185,-7,31815.2250,450000,0.0000,0.0000,0.0000,0.0000,2250.0000,2250.0000,2250.0000,26926.4250,31460.0850,31460.0850,0.0000,0,0.0000,0.0000,30.0000,Active,0,0
3,1389973,337855,-4,236572.1100,225000,2250.0000,2250.0000,0.0000,0.0000,11795.7600,11925.0000,11925.0000,224949.2850,233048.9700,233048.9700,1.0000,1,0.0000,0.0000,10.0000,Active,0,0
4,1891521,126868,-1,453919.4550,450000,0.0000,11547.0000,0.0000,11547.0000,22924.8900,27000.0000,27000.0000,443044.3950,453919.4550,453919.4550,0.0000,1,0.0000,1.0000,101.0000,Active,0,0


In [110]:
credit_card_records_per_client = (
    credit_card.groupby("SK_ID_CURR")["SK_ID_PREV"]
    .count()
    .sort_values(ascending=False)
)

credit_card_records_per_client.describe()

count   103558.0000
mean        37.0837
std         33.4836
min          1.0000
25%         10.0000
50%         22.0000
75%         75.0000
max        192.0000
Name: SK_ID_PREV, dtype: float64

### 13.1 Credit card aggregations

Создадим агрегированные признаки по истории кредитных карт клиента.

Для каждого клиента посчитаем:

- количество записей;
- количество уникальных кредитных карт;
- средний и максимальный баланс;
- средний кредитный лимит;
- среднюю задолженность;
- среднюю и максимальную просрочку;
- средние суммы снятий и платежей.

In [111]:
credit_card_agg = credit_card.groupby("SK_ID_CURR").agg(
    CC_RECORDS_COUNT=("SK_ID_PREV", "count"),
    CC_PREV_UNIQUE_COUNT=("SK_ID_PREV", "nunique"),
    CC_MONTHS_BALANCE_MEAN=("MONTHS_BALANCE", "mean"),
    CC_MONTHS_BALANCE_MIN=("MONTHS_BALANCE", "min"),
    CC_MONTHS_BALANCE_MAX=("MONTHS_BALANCE", "max"),
    CC_AMT_BALANCE_MEAN=("AMT_BALANCE", "mean"),
    CC_AMT_BALANCE_MAX=("AMT_BALANCE", "max"),
    CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
    CC_AMT_CREDIT_LIMIT_ACTUAL_MAX=("AMT_CREDIT_LIMIT_ACTUAL", "max"),
    CC_AMT_DRAWINGS_CURRENT_MEAN=("AMT_DRAWINGS_CURRENT", "mean"),
    CC_AMT_DRAWINGS_CURRENT_MAX=("AMT_DRAWINGS_CURRENT", "max"),
    CC_AMT_PAYMENT_CURRENT_MEAN=("AMT_PAYMENT_CURRENT", "mean"),
    CC_AMT_PAYMENT_CURRENT_SUM=("AMT_PAYMENT_CURRENT", "sum"),
    CC_AMT_RECEIVABLE_PRINCIPAL_MEAN=("AMT_RECEIVABLE_PRINCIPAL", "mean"),
    CC_AMT_RECIVABLE_MEAN=("AMT_RECIVABLE", "mean"),
    CC_AMT_TOTAL_RECEIVABLE_MEAN=("AMT_TOTAL_RECEIVABLE", "mean"),
    CC_CNT_DRAWINGS_CURRENT_MEAN=("CNT_DRAWINGS_CURRENT", "mean"),
    CC_CNT_DRAWINGS_CURRENT_MAX=("CNT_DRAWINGS_CURRENT", "max"),
    CC_SK_DPD_MEAN=("SK_DPD", "mean"),
    CC_SK_DPD_MAX=("SK_DPD", "max"),
    CC_SK_DPD_DEF_MEAN=("SK_DPD_DEF", "mean"),
    CC_SK_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
).reset_index()

credit_card_agg.shape

(103558, 23)

In [112]:
credit_card_cat = pd.get_dummies(
    credit_card[["SK_ID_CURR", "NAME_CONTRACT_STATUS"]],
    columns=["NAME_CONTRACT_STATUS"],
    dummy_na=True
)

credit_card_cat_agg = credit_card_cat.groupby("SK_ID_CURR").sum().reset_index()

credit_card_cat_agg = credit_card_cat_agg.rename(
    columns={
        col: f"CC_{col}"
        for col in credit_card_cat_agg.columns
        if col != "SK_ID_CURR"
    }
)

credit_card_features = credit_card_agg.merge(
    credit_card_cat_agg,
    on="SK_ID_CURR",
    how="left"
)

credit_card_features.shape

(103558, 31)

In [113]:
credit_card_features.isna().mean().sort_values(ascending=False).head(10)

CC_AMT_PAYMENT_CURRENT_MEAN             0.3036
SK_ID_CURR                              0.0000
CC_AMT_TOTAL_RECEIVABLE_MEAN            0.0000
CC_NAME_CONTRACT_STATUS_Signed          0.0000
CC_NAME_CONTRACT_STATUS_Sent proposal   0.0000
CC_NAME_CONTRACT_STATUS_Refused         0.0000
CC_NAME_CONTRACT_STATUS_Demand          0.0000
CC_NAME_CONTRACT_STATUS_Completed       0.0000
CC_NAME_CONTRACT_STATUS_Approved        0.0000
CC_NAME_CONTRACT_STATUS_Active          0.0000
dtype: float64

In [114]:
np.isinf(
    credit_card_features.select_dtypes(include=["int64", "float64"])
).sum().sort_values(ascending=False).head(10)

SK_ID_CURR                               0
CC_AMT_TOTAL_RECEIVABLE_MEAN             0
CC_NAME_CONTRACT_STATUS_Signed           0
CC_NAME_CONTRACT_STATUS_Sent proposal    0
CC_NAME_CONTRACT_STATUS_Refused          0
CC_NAME_CONTRACT_STATUS_Demand           0
CC_NAME_CONTRACT_STATUS_Completed        0
CC_NAME_CONTRACT_STATUS_Approved         0
CC_NAME_CONTRACT_STATUS_Active           0
CC_SK_DPD_DEF_MAX                        0
dtype: int64

### 13.2 Merge credit card features with application

Теперь присоединим агрегированные признаки из `credit_card_balance` к основной таблице.

Эти признаки описывают историю использования кредитных карт клиента: баланс, кредитный лимит, задолженность, платежи, снятия и просрочки.

In [115]:
application_shape_before_credit_card = application.shape

application = application.merge(
    credit_card_features,
    on="SK_ID_CURR",
    how="left"
)

application_shape_before_credit_card, application.shape

((356255, 295), (356255, 325))

In [116]:
credit_card_feature_cols = [
    col for col in credit_card_features.columns
    if col != "SK_ID_CURR"
]

application[credit_card_feature_cols].isna().all(axis=1).mean()

np.float64(0.7093149569830599)

In [117]:
application[
    [
        "SK_ID_CURR",
        "DATASET",
        "CC_RECORDS_COUNT",
        "CC_PREV_UNIQUE_COUNT",
        "CC_AMT_BALANCE_MEAN",
        "CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN",
        "CC_SK_DPD_MEAN",
        "CC_SK_DPD_MAX",
    ]
].head()

,SK_ID_CURR,DATASET,CC_RECORDS_COUNT,CC_PREV_UNIQUE_COUNT,CC_AMT_BALANCE_MEAN,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_SK_DPD_MEAN,CC_SK_DPD_MAX
0,100002,train,NaN,NaN,NaN,NaN,NaN,NaN
1,100003,train,NaN,NaN,NaN,NaN,NaN,NaN
2,100004,train,NaN,NaN,NaN,NaN,NaN,NaN
3,100006,train,6.0000,1.0000,0.0000,270000.0000,0.0000,0.0000
4,100007,train,NaN,NaN,NaN,NaN,NaN,NaN


Вывод:

Признаки из `credit_card_balance` успешно агрегированы до уровня клиента и объединены с основной таблицей.

Так как кредитные карты есть не у всех клиентов, часть строк содержит пропуски по этим признакам. Это ожидаемо.

## 14. Final dataset checks

Мы собрали итоговую таблицу признаков на уровне клиента.

В неё вошли:

- признаки из основной таблицы `application`;
- агрегаты из `bureau`;
- агрегаты из `bureau_balance`;
- агрегаты из `previous_application`;
- агрегаты из `POS_CASH_balance`;
- агрегаты из `installments_payments`;
- агрегаты из `credit_card_balance`.

Теперь проверим итоговый размер, дубликаты, пропуски и бесконечные значения.

In [118]:
application.shape

(356255, 325)

In [119]:
application["SK_ID_CURR"].duplicated().sum()

np.int64(0)

In [120]:
application["DATASET"].value_counts()

DATASET
train    307511
test      48744
Name: count, dtype: int64

In [121]:
application.groupby("DATASET")["TARGET"].agg(["count", "mean"])

,count,mean
DATASET,,
test,0,NaN
train,307511,0.0807


In [122]:
missing_summary = (
    application
    .isna()
    .mean()
    .sort_values(ascending=False)
)

missing_summary.head(30)

CC_AMT_PAYMENT_CURRENT_MEAN             0.7976
CC_NAME_CONTRACT_STATUS_nan             0.7093
CC_AMT_RECIVABLE_MEAN                   0.7093
CC_RECORDS_COUNT                        0.7093
CC_PREV_UNIQUE_COUNT                    0.7093
CC_MONTHS_BALANCE_MEAN                  0.7093
CC_MONTHS_BALANCE_MIN                   0.7093
CC_MONTHS_BALANCE_MAX                   0.7093
CC_AMT_BALANCE_MEAN                     0.7093
CC_AMT_BALANCE_MAX                      0.7093
CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN         0.7093
CC_AMT_CREDIT_LIMIT_ACTUAL_MAX          0.7093
CC_AMT_DRAWINGS_CURRENT_MAX             0.7093
CC_AMT_PAYMENT_CURRENT_SUM              0.7093
CC_AMT_RECEIVABLE_PRINCIPAL_MEAN        0.7093
CC_AMT_DRAWINGS_CURRENT_MEAN            0.7093
CC_AMT_TOTAL_RECEIVABLE_MEAN            0.7093
CC_NAME_CONTRACT_STATUS_Active          0.7093
CC_NAME_CONTRACT_STATUS_Signed          0.7093
CC_CNT_DRAWINGS_CURRENT_MEAN            0.7093
CC_NAME_CONTRACT_STATUS_Refused         0.7093
CC_NAME_CONTR

In [123]:
numeric_application = application.select_dtypes(include=["int64", "float64"])

inf_summary = (
    np.isinf(numeric_application)
    .sum()
    .sort_values(ascending=False)
)

inf_summary.head(20)

SK_ID_CURR                                 0
PREV_NAME_CONTRACT_STATUS_Approved         0
PREV_NAME_CONTRACT_TYPE_Revolving loans    0
PREV_NAME_CONTRACT_TYPE_Consumer loans     0
PREV_NAME_CONTRACT_TYPE_Cash loans         0
PREV_NAME_CONTRACT_STATUS_nan              0
PREV_NAME_CONTRACT_STATUS_Unused offer     0
PREV_NAME_CONTRACT_STATUS_Refused          0
PREV_NAME_CONTRACT_STATUS_Canceled         0
PREV_CREDIT_TO_APPLICATION_RATIO           0
PREV_AMT_DOWN_PAYMENT_MEAN                 0
PREV_CNT_PAYMENT_MAX                       0
PREV_CNT_PAYMENT_MEAN                      0
PREV_DAYS_DECISION_MAX                     0
PREV_DAYS_DECISION_MIN                     0
PREV_DAYS_DECISION_MEAN                    0
PREV_RATE_DOWN_PAYMENT_MEAN                0
PREV_HOUR_APPR_PROCESS_START_MEAN          0
PREV_NAME_CONTRACT_TYPE_XNA                0
PREV_NAME_CONTRACT_TYPE_nan                0
dtype: int64

## 15. Split train and test

После создания всех признаков разделим объединённую таблицу обратно на train и test.

- `train_features` содержит `TARGET`;
- `test_features` не должен содержать `TARGET`;
- техническую колонку `DATASET` удалим.

In [124]:
train_features = application[application["DATASET"] == "train"].copy()
test_features = application[application["DATASET"] == "test"].copy()

train_features.shape, test_features.shape

((307511, 325), (48744, 325))

In [125]:
train_features = train_features.drop(columns=["DATASET"])
test_features = test_features.drop(columns=["DATASET", "TARGET"])

train_features.shape, test_features.shape

((307511, 324), (48744, 323))

In [126]:
train_features["TARGET"].notna().all(), "TARGET" in test_features.columns

(np.True_, False)

In [127]:
train_features["SK_ID_CURR"].duplicated().sum(), test_features["SK_ID_CURR"].duplicated().sum()

(np.int64(0), np.int64(0))

## 16. Save feature datasets

Сохраним итоговые признаки в папку `data/processed`.

На выходе получим:

- `train_features.parquet`;
- `test_features.parquet`;
- `feature_columns.csv`.

Эти файлы дальше будут использоваться на этапе preprocessing/modeling.

In [128]:
train_features_path = PROCESSED_DATA_DIR / "train_features.parquet"
test_features_path = PROCESSED_DATA_DIR / "test_features.parquet"
feature_columns_path = PROCESSED_DATA_DIR / "feature_columns.csv"

train_features.to_parquet(train_features_path, index=False)
test_features.to_parquet(test_features_path, index=False)

feature_columns = pd.DataFrame({
    "feature": train_features.drop(columns=["TARGET"]).columns
})

feature_columns.to_csv(feature_columns_path, index=False)

train_features_path, test_features_path, feature_columns_path

(PosixPath('/Users/artem/PycharmProjects/credit-risk-scoring-service/data/processed/train_features.parquet'),
 PosixPath('/Users/artem/PycharmProjects/credit-risk-scoring-service/data/processed/test_features.parquet'),
 PosixPath('/Users/artem/PycharmProjects/credit-risk-scoring-service/data/processed/feature_columns.csv'))

In [129]:
train_features_path.exists(), test_features_path.exists(), feature_columns_path.exists()

(True, True, True)

## 17. Final conclusions

В этом ноутбуке был выполнен feature engineering для задачи кредитного скоринга.

Были созданы признаки из основной таблицы клиента и агрегированные признаки из исторических таблиц.

Итоговые датасеты сохранены в `data/processed`:

- `train_features.parquet`;
- `test_features.parquet`;
- `feature_columns.csv`.

Следующий этап проекта — preprocessing и baseline modeling.